# Can Chair44 (R44) fill space without ever repeating?

**Run it yourself.** Choose **Runtime → Run all** in Colab. Then rotate the tile,
inspect an overlap, change a shell index, or deliberately break a certificate.
Code is folded; expand it whenever you want to inspect the calculation.

> “$Q$ admits a tiling of $\mathbb{R}^3$.”
> “Every tiling $T$ by $Q$ has $\operatorname{Per}(T)=\{0\}$.”
> “Every tiling $T$ by $Q$ has $|\operatorname{Sym}(T)|\le24$.”
> — paper §3 (Existence) and §7 (paper macros expanded for display; congruent copies include reflections).

**This notebook is a reader utility, not evidence for the theorem.** Its finite
computations are **T2**. **T1** means Lean with standard axioms only; **T1n** adds
named compiler hooks; **T3** means written proof or cited import. Pictures use
floats for display only. The formal build is a separate action at the end.

A fresh local Jupyter run on host `ai-box` took 98 seconds on 2026-09-11
(Python 3.12). Your own runtime, host and date are recorded below. A hosted
Colab timing is not yet available. The setup fetches a fixed source snapshot,
not today's main.

In [ ]:
# This import is expanded by build.py so the downloadable notebook stands alone.
"""U4 presentation and session support; embedded into the generated notebook.

Geometric predicates are executed from the pinned proof packets. Coordinates
are converted to floats only at the final display boundary.
"""
import contextlib
import copy
import datetime
from fractions import Fraction
import gzip
import hashlib
import html
import io
import json
import os
from pathlib import Path
import platform
import runpy
import subprocess
import sys
import tarfile
import tempfile
import time

from IPython.display import HTML, Markdown, display


class Session:
    def __init__(self, pin, claim_map):
        self.pin = pin
        self.claim_map = claim_map
        self.directory = Path(tempfile.mkdtemp(prefix="r44-reader-"))
        self.source = self.directory / "source"
        self.source.mkdir()
        self.receipt = {
            "notice": "Unsigned session record. Records only this session's executions; this utility is not evidence for the theorem.",
            "commit": pin["source_commit"], "mathematical_baseline_commit": pin["mathematical_baseline_commit"],
            "canonical_sha256": {}, "python": sys.version, "platform": platform.platform(),
            "host": "Google Colab" if "google.colab" in sys.modules else platform.node(),
            "date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "cells": {}, "lean_ran": False, "replay_report": None,
        }
        self.receipt_path = self.directory / "receipt.json"
        self.save()

    def save(self):
        self.receipt_path.write_text(json.dumps(self.receipt, indent=2) + "\n")

    def run(self, name, action, tier="T2"):
        start = time.monotonic()
        artifact_keys = {
            "finite_replay": "replay_report", "collision_witness": "collision_selection",
            "parent_certificate": "parent_selection", "atlas_equality": "atlas_difference",
            "mutation_controls": "mutation_controls", "periodic_controls": "periodic_controls",
            "companion_census": "companion_census", "selected_mutation": "selected_mutation",
        }
        if name in artifact_keys:
            self.receipt.pop(artifact_keys[name], None)
        record = {"tier": tier, "status": "running"}
        self.receipt["cells"][name] = record
        self.save()
        try:
            result = action()
            record["status"] = "PASS"
            return result
        except BaseException as exc:
            record.update(status="interrupted" if isinstance(exc, KeyboardInterrupt) else "FAIL", error=str(exc))
            raise
        finally:
            record["seconds"] = round(time.monotonic() - start, 3)
            self.save()

    def command(self, args, cwd=None, log="command", timeout=240):
        proc = subprocess.run(args, cwd=cwd or self.source, text=True,
                              capture_output=True, timeout=timeout)
        (self.directory / (log + ".log")).write_text(proc.stdout + proc.stderr)
        if proc.returncode:
            raise RuntimeError(f"{log} exited {proc.returncode}:\n{proc.stderr[-3000:]}")
        return proc.stdout

    def prepare(self):
        def action():
            repo = os.environ.get("R44_SOURCE_REPO")
            if repo is None:
                repo = str(self.directory / "git")
                self.command(["git", "init", "--quiet", repo], log="git-init")
                self.command(["git", "-C", repo, "remote", "add", "origin",
                              "https://github.com/ioannist/six-birds-tiles.git"], log="git-remote")
                self.command(["git", "-C", repo, "fetch", "--depth=1", "--filter=blob:none",
                              "origin", self.pin["source_commit"]], log="git-fetch", timeout=600)
            commit = subprocess.check_output(["git", "-C", repo, "rev-parse",
                                               self.pin["source_commit"] + "^{commit}"], text=True).strip()
            if commit != self.pin["source_commit"]:
                raise ValueError("Source commit mismatch")
            paths = ["solid", "certificates", "verify", "simulations", "viewer",
                     "paper/tex", "paper/CLAIMS_LEDGER.md", "lean/R44/build_axioms.log",
                     "lean/R44/negative_control.log", "lean/R44/HYPOTHESES.md", "lean/R44/AXIOMS.md"]
            archive = self.directory / "source.tar"
            with archive.open("wb") as out:
                subprocess.run(["git", "-C", repo, "archive", commit, "--", *paths], stdout=out, check=True)
            forbidden = {".git", ".codex", "offgit", "history", "archive", ".lake", "review", "TILE_DISCOVERY_THREAD.txt"}
            with tarfile.open(archive) as tar:
                for member in tar:
                    parts = Path(member.name).parts
                    if any(part in forbidden for part in parts):
                        continue
                    if member.name.startswith("/") or ".." in parts or not (member.isfile() or member.isdir()):
                        raise ValueError("Unsafe source archive member: " + member.name)
                    dest = self.source / member.name
                    if member.isdir():
                        dest.mkdir(parents=True, exist_ok=True)
                    else:
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        dest.write_bytes(tar.extractfile(member).read())
            archive.unlink()
            self.check_hashes()
            expected_sources = {"paper/CLAIMS_LEDGER.md": self.claim_map["ledger_sha256"],
                                "lean/R44/build_axioms.log": self.claim_map["axiom_log_sha256"],
                                **self.claim_map["tex_sha256"]}
            for path, digest in expected_sources.items():
                if hashlib.sha256((self.source/path).read_bytes()).hexdigest() != digest:
                    raise ValueError("Evidence-map source mismatch: " + path)
            # Only immutable source files are copied; output-producing checks run here.
            self.work = self.directory / "work"
            import shutil
            shutil.copytree(self.source, self.work)
            display(Markdown("**Source snapshot** `" + commit + "`"))
            display(Markdown("\n".join(f"- `{path}`: `{digest}`" for path, digest in self.receipt["canonical_sha256"].items())))
        self.run("setup", action, "T2 (file identity)")

    def check_hashes(self):
        for path, expected in self.pin["canonical_sha256"].items():
            actual = hashlib.sha256((self.source / path).read_bytes()).hexdigest()
            if actual != expected:
                raise ValueError("Canonical digest mismatch: " + path)
            self.receipt["canonical_sha256"][path] = actual
        self.save()

    def replay(self):
        def action():
            self.command([sys.executable, "verify/replay.py"], cwd=self.work, log="replay")
            report = json.loads((self.work / "verify/replay_report.json").read_text())
            self.receipt["replay_report"] = report
            if report["status"] != "PASS":
                raise ValueError("Replay did not pass")
            if self.receipt["canonical_sha256"] != report["canonical_sha256"]:
                raise ValueError("Replay digest mismatch")
            display(Markdown("**T2 — finite replay** · " + str(report["seconds"]) + " seconds in this session"))
            display(Markdown("\n".join("- `" + r["packet"] + "`: exit " + str(r["returncode"]) for r in report["steps"])))
        self.run("finite_replay", action)

    def module(self, relative):
        # Some canonical scripts execute on import and produce JSON files.
        capture = io.StringIO()
        with contextlib.redirect_stdout(capture):
            result = runpy.run_path(str(self.work / relative))
        (self.directory / (Path(relative).stem + ".log")).write_text(capture.getvalue())
        return result

    def companions(self):
        def action():
            output = self.directory / "fresh-alignment.json"
            output.unlink(missing_ok=True)
            self.command([sys.executable, "verify/packets/r44_unrestricted_alignment/src/verify_alignment.py",
                          "--output", str(output)], cwd=self.work, log="alignment")
            report = json.loads(output.read_text())
            if report["status"] != "PASS":
                raise ValueError("Alignment checker did not pass")
            self.receipt["companion_census"] = report
            display(HTML("<details><summary>T2 — inspect the fresh alignment report</summary><pre>" + html.escape(json.dumps(report, indent=2)) + "</pre></details>"))
        self.run("companion_census", action)

    def viewer(self):
        text = (self.source / "viewer/index.html").read_text()
        for name in ["vendor/three.min.js", "r44_data.js"]:
            code = (self.source / "viewer" / name).read_text().replace("</script", "<\\/script")
            text = text.replace('<script src="' + name + '"></script>', "<script>" + code + "</script>")
        data_text = (self.source / "viewer/r44_data.js").read_text()
        for path in ["solid/r44_solid.json", "certificates/candidate_certificate.json"]:
            if self.pin["canonical_sha256"][path] not in data_text:
                raise ValueError("Viewer source hash mismatch")
        self.iframe(text, "R44 viewer")

    def iframe(self, document, title):
        display(HTML('<iframe title="' + html.escape(title) + '" sandbox="allow-scripts" '
                     'style="width:100%;height:660px;border:0" srcdoc="' + html.escape(document, quote=True) + '"></iframe>'))

    def mesh_display(self, poses, box=None):
        """Display only: input poses are exact rational row-matrix isometries."""
        solid = json.loads((self.source / "solid/r44_solid.json").read_text())
        vertices = [list(map(Fraction, p)) for p in solid["vertices"]]
        meshes = []
        for matrix, translation in poses:
            transformed = [[sum(Fraction(matrix[i][j])*v[j] for j in range(3)) + Fraction(translation[i])
                            for i in range(3)] for v in vertices]
            meshes.append([float(x) for tri in solid["triangles"] for k in tri for x in transformed[k]])
        payload = json.dumps({"meshes": meshes, "box": [[float(x) for x in side] for side in box] if box else None})
        three = (self.source / "viewer/vendor/three.min.js").read_text().replace("</script", "<\\/script")
        script = """
const data=PAYLOAD, scene=new THREE.Scene(); scene.background=new THREE.Color('#1b2540');
const camera=new THREE.PerspectiveCamera(40,innerWidth/innerHeight,.01,1000);
const renderer=new THREE.WebGLRenderer({antialias:true}); renderer.setSize(innerWidth,innerHeight); document.body.append(renderer.domElement);
scene.add(new THREE.AmbientLight(0xffffff,.8)); const light=new THREE.DirectionalLight(0xffffff,.8); light.position.set(5,8,10); scene.add(light);
const group=new THREE.Group(); scene.add(group);
data.meshes.forEach((positions,i)=>{const g=new THREE.BufferGeometry();g.setAttribute('position',new THREE.Float32BufferAttribute(positions,3));g.computeVertexNormals();group.add(new THREE.Mesh(g,new THREE.MeshPhongMaterial({color:[0x5aa9e6,0xe8563f,0xf3b53a][i%3],transparent:true,opacity:.4,side:THREE.DoubleSide})));});
if(data.box){const [lo,hi]=data.box;const g=new THREE.BoxGeometry(...hi.map((x,i)=>x-lo[i]));const b=new THREE.Mesh(g,new THREE.MeshBasicMaterial({color:0xf3b53a}));b.position.set(...hi.map((x,i)=>(x+lo[i])/2));group.add(b);}
const bounds=new THREE.Box3().setFromObject(group),center=bounds.getCenter(new THREE.Vector3());group.position.sub(center);
const pivot=new THREE.Group();scene.add(pivot);pivot.add(group);
const radius=Math.max(2,bounds.getSize(new THREE.Vector3()).length());camera.position.set(radius*.7,radius*.5,radius);camera.lookAt(0,0,0);
let down=false,x=0;renderer.domElement.onpointerdown=e=>{down=true;x=e.clientX;renderer.domElement.setPointerCapture(e.pointerId)};renderer.domElement.onpointerup=()=>down=false;renderer.domElement.onpointermove=e=>{if(down){pivot.rotation.y+=(e.clientX-x)*.01;x=e.clientX}};
renderer.domElement.onwheel=e=>{e.preventDefault();camera.position.multiplyScalar(e.deltaY>0?1.08:.92)};
onresize=()=>{camera.aspect=innerWidth/innerHeight;camera.updateProjectionMatrix();renderer.setSize(innerWidth,innerHeight)};
(function loop(){requestAnimationFrame(loop);renderer.render(scene,camera)})();
""".replace("PAYLOAD", payload)
        self.iframe('<html><body style="margin:0;overflow:hidden"><div style="position:absolute;color:white;padding:12px;font:14px sans-serif">Display only · floats · true geometry · drag to rotate, scroll to zoom</div><script>' + three + '</script><script>' + script + '</script></body></html>', "Exact certificate data displayed in 3D")

    def collision(self, record_index=0, partner_index=0):
        def action():
            if not hasattr(self, "core"):
                self.core = self.module("verify/packets/r44_unrestricted_alignment/src/replay_core_boxes.py")
            with gzip.open(self.source / "certificates/collision_core_witnesses.jsonl.gz", "rt") as stream:
                records = (json.loads(line) for line in stream)
                record = next((r for i, r in enumerate(records) if i == record_index), None)
            if record is None:
                raise ValueError("Record index out of range")
            g, t = map(tuple, record["normalized_other"])
            h, u, ia, ib, claimed_lo, claimed_hi = record["overlaps"][partner_index]
            h, u = tuple(h), tuple(u)
            a = self.core["selected_box"](g,t,ia); b = self.core["selected_box"](h,u,ib)
            lo = tuple(max(a[i], b[i]) for i in range(3)); hi = tuple(min(a[i]+8,b[i]+8) for i in range(3))
            self.core["check"](lo == tuple(claimed_lo) and hi == tuple(claimed_hi), "wrong displayed witness")
            lower = tuple(Fraction(50*x+4,400) for x in lo); upper = tuple(Fraction(50*x-4,400) for x in hi)
            self.core["check"](min(upper[i]-lower[i] for i in range(3)) >= Fraction(42,400), "invalid retained core")
            def pose(frame, shift):
                # Packet frames store the signed image of each input basis vector.
                matrix = [[(1 if frame[j]>0 else -1) if abs(frame[j]) == i+1 else 0 for j in range(3)] for i in range(3)]
                return matrix, [Fraction(x,8) for x in shift]
            result = {"record": record_index, "partner": partner_index, "lower": list(map(str,lower)), "upper": list(map(str,upper))}
            self.receipt["collision_selection"] = result
            display(Markdown("**T2 — retained-core collision witness** · yellow box; two implicated copies in blue and coral."))
            display(Markdown("Exact lower corner: `"+str(result["lower"])+"`; upper corner: `"+str(result["upper"])+"`."))
            self.mesh_display([pose(g,t),pose(h,u)], (lower,upper))
        self.run("collision_witness", action)

    def parent(self, shell_index=0):
        def action():
            data = json.loads((self.source / "certificates/candidate_certificate.json").read_text())
            roles = data["role_options"][shell_index]
            if len(roles) != 1:
                raise ValueError("Expected one recorded parent role")
            def pose(p):
                (perm, signs), shift = p
                return [[signs[i] if j==perm[i] else 0 for j in range(3)] for i in range(3)], shift
            display(Markdown(f"**T2 — certificate shell {shell_index}** · recorded role {roles[0]}. This displays the certificate; interactive recognition of a reader's own patch is deferred to U2."))
            self.receipt["parent_selection"] = {"shell":shell_index,"roles":roles,"contacts":data["solutions"][shell_index]}
            display(Markdown("Contact indices: `"+str(data["solutions"][shell_index])+"`"))
            poses = [pose([[[0,1,2],[1,1,1]],[0,0,0]])] + [pose(data["legal_contacts"][i]) for i in data["solutions"][shell_index]]
            self.mesh_display(poses)
        self.run("parent_certificate", action)

    def atlas(self):
        def action():
            namespace = self.module("verify/packets/einstein_macrostate/src/verify.py")
            fine, coarse = namespace["legal"], namespace["coarse"]
            result = {"fine_count":len(fine),"coarse_count":len(coarse),
                      "missing":sorted(fine-coarse),"additional":sorted(coarse-fine)}
            self.receipt["atlas_difference"] = result
            if result["missing"] or result["additional"]:
                raise ValueError("Contact sets differ")
            display(Markdown("**T2 — rescaled parent atlas versus fine atlas**\n\nMissing contacts: **0**. Additional contacts: **0**."))
            display(HTML("<details><summary>Inspect both sets and their differences</summary><pre>"+html.escape(json.dumps({**result,"fine":sorted(fine),"coarse":sorted(coarse)},indent=2))+"</pre></details>"))
        self.run("atlas_equality", action)

    def mutations(self):
        def action():
            root = self.work / "verify/packets/r44_unrestricted_alignment"
            self.command([sys.executable,str(root / "src/test_mutations.py")], cwd=root, log="mutations")
            report = json.loads((root / "results/mutation_controls.json").read_text())
            if len(report["tests"]) != 6 or not all(t["rejected"] for t in report["tests"]):
                raise ValueError("Mutation control failed")
            self.receipt["mutation_controls"] = report
            self.check_hashes()
            display(Markdown("**T2 — all six temporary corruptions rejected.** Canonical inputs retain their original hashes; temporary variants were removed."))
            return report
        return self.run("mutation_controls", action)

    def mutation(self, name="missing_triangle", index=0):
        """Apply the existing six corruption recipes at a reader-selected index.

        Only input construction is adapted from test_mutations.py; the unchanged
        verify_alignment.py judges the result. No geometric predicate is added.
        """
        def action():
            choices = {
                "missing_triangle": ("solid/r44_solid.json", "--solid", "triangles", "nonmanifold mesh edge"),
                "reversed_triangle": ("solid/r44_solid.json", "--solid", "triangles", "unpaired oriented edge"),
                "changed_geometric_apex": ("solid/r44_solid.json", "--solid", "patches", "wrong feature height"),
                "missing_rejection_witness": ("certificates/companion_collision_certificate.json", "--collisions", "witnesses", "duplicate/missing rejection witness"),
                "wrong_companion_quantifier": ("certificates/companion_collision_certificate.json", "--collisions", "witnesses", "incomplete partner quantifier"),
                "missing_registered_contact": ("certificates/candidate_certificate.json", "--registered", "legal_contacts", "survivors not identical to registered 44 atlas"),
            }
            path, flag, field, diagnostic = choices[name]
            data = json.loads((self.source/path).read_text())
            if not 0 <= index < len(data[field]):
                raise ValueError(f"Choose an index between 0 and {len(data[field])-1}")
            if name.startswith("missing_"):
                data[field].pop(index)
            elif name == "reversed_triangle":
                tri = data[field][index]; tri[0], tri[1] = tri[1], tri[0]
            elif name == "wrong_companion_quantifier":
                data[field][index]["possible_partners"] += 1
            else:
                apex = tuple(map(Fraction, data["patches"][index]["apex"]))
                k = next(i for i,v in enumerate(data["vertices"]) if tuple(map(Fraction,v)) == apex)
                data["vertices"][k][0] = str(Fraction(data["vertices"][k][0]) + Fraction(1,100000))
            with tempfile.TemporaryDirectory(dir=self.directory, prefix="mutation-") as tmp:
                altered = Path(tmp)/"altered.json"; altered.write_text(json.dumps(data))
                proc = subprocess.run([sys.executable, str(self.work/"verify/packets/r44_unrestricted_alignment/src/verify_alignment.py"),
                                       flag, str(altered), "--output",str(Path(tmp)/"result.json")], capture_output=True,text=True,timeout=90)
                diagnostics = [diagnostic, "off-center pyramid apex"] if name == "changed_geometric_apex" else [diagnostic]
                actual_diagnostic = next((d for d in diagnostics if ("ValueError: " + d) in proc.stderr), None)
                if proc.returncode == 0 or actual_diagnostic is None:
                    raise RuntimeError("Unexpected mutation result; this is not a successful rejection control:\n" + proc.stderr[-1200:])
                diagnostic = actual_diagnostic
            self.check_hashes()
            self.receipt["selected_mutation"] = {"name":name,"index":index,"diagnostic":diagnostic,"rejected":True,"temporary_copy_removed":True}
            display(Markdown("**T2 — selected mutation rejected:** `"+name+"`, index "+str(index)+".\n\n`ValueError: "+diagnostic+"`\n\nTemporary change removed; canonical hashes unchanged."))
        self.run("selected_mutation", action)

    def controls(self):
        def action():
            try:
                import pysat
            except ImportError:
                self.command([sys.executable,"-m","pip","install","python-sat==1.9.dev15"], log="install-python-sat", timeout=600)
                import pysat
            output = self.directory / "periodic-controls.json"
            self.command([sys.executable,"simulations/periodicity_search.py","--controls","--control-index","7","--out",str(output)], cwd=self.work, log="periodic-controls", timeout=180)
            report = json.loads(output.read_text())
            if len(report["runs"]) != 2 or not all(r["control_passed"] and r["torus"]["sat_witness"] is not None for r in report["runs"]):
                raise ValueError("Periodic controls failed")
            self.receipt["periodic_controls"] = report
            self.receipt["python_sat_version"] = pysat.__version__
            display(Markdown("**T2-adjacent — evidence only, used nowhere in the proof.**"))
            for result in report["runs"]:
                display(Markdown("Periodic control: **"+result["tile"]+"**"))
                display(HTML("<details><summary>Inspect periodic witness</summary><pre>"+html.escape(json.dumps(result["torus"]["sat_witness"],indent=2))+"</pre></details>"))
        self.run("periodic_controls", action, "T2-adjacent (paper §8.1)")

    def sources(self):
        self.run("source_reading", self._sources, "T1/T1n/T3 source records; not executed")

    def _sources(self):
        display(Markdown("**T1 / T1n / T3 — source statements and supplied axiom records. No Lean build runs in this notebook.**"))
        rows = {r["id"]:r for r in self.claim_map["rows"]}
        groups = [
            ("Existence and exhaustion", ["T1", "R8", "R10"], ["R44.existence", "R44.r44_einstein"]),
            ("Registration", ["T10", "A14"], ["R44.unrestricted_alignment_holds"]),
            ("Iterated parents and coarsening", ["T8", "R6", "R7"], ["R44.geometric_hierarchy_unique", "R44.carrier_hierarchy", "R44.parent_atlas_eq_fine"]),
            ("Period halving", ["T2", "T13"], ["R44.period_halving", "R44.no_period", "R44.r44_einstein"]),
        ]
        for title, ids, names in groups:
            display(Markdown("### " + title))
            for claim in ids:
                row = rows[claim]
                display(Markdown("**" + claim + " · " + row["ledger_tier"] + "** — quoted ledger statement:\n\n> " + row["statement"] + "\n\nPaper sections: " + str(row["paper_sections"])))
            for name in names:
                record = self.claim_map["axiom_records"].get(name)
                if record is None:
                    raise ValueError("Missing axiom map declaration: " + name)
                display(HTML('<details><summary>'+html.escape(name)+' — supplied axiom record</summary><pre>'+html.escape(json.dumps(record,indent=2))+'</pre></details>'))
        display(HTML('<details><summary>Complete generated evidence map</summary><pre>'+html.escape(json.dumps(self.claim_map,indent=2))+'</pre></details>'))
        for path in ["paper/tex/sec3_finding.tex","paper/tex/sec5_registration.tex","paper/tex/sec6_hierarchy.tex","paper/tex/sec7_aperiodicity.tex"]:
            display(HTML('<details><summary>Quoted proof source: '+html.escape(path)+'</summary><pre>'+html.escape((self.source/path).read_text())+'</pre></details>'))
        for path in ["lean/R44/build_axioms.log", "lean/R44/negative_control.log"]:
            display(HTML('<details><summary>Supplied record, not a fresh build: '+html.escape(path)+'</summary><pre>'+html.escape((self.source/path).read_text())+'</pre></details>'))

    def download(self):
        self.check_hashes()
        for name in ["setup","finite_replay","viewer","collision_witness","companion_census","parent_certificate","atlas_equality","mutation_controls","selected_mutation","periodic_controls","source_reading"]:
            self.receipt["cells"].setdefault(name,{"status":"skipped"})
        self.save()
        import base64
        data = base64.b64encode(self.receipt_path.read_bytes()).decode()
        display(HTML('<a download="r44-receipt.json" href="data:application/json;base64,'+data+'">Download your session receipt</a>'))
        display(Markdown("Unsigned session record · `lean_ran: false` · " + str(self.receipt_path)))

# These assignments are filled from reader/pin.json and its generated evidence map.
PIN = {'source_commit': '838bca514679b2531d31f4e0dfd66641269e2a8c', 'mathematical_baseline_commit': 'd90313a717994936f254990f88d2624bbcce5bcd', 'canonical_sha256': {'solid/r44_solid.json': 'f320d7a0c2d784a3eb29f001dc035808f67591d9ea8d3dd45949e66442f0ed55', 'certificates/candidate_certificate.json': '44e9b3f6048497de02d1be72d0c80ddc94aaaf235382138c7468ffdbc5de5ff2', 'certificates/companion_collision_certificate.json': '36e89e2fe81f4788e7edea2a68fef013557e5b73f5a16ac76348fe241b068b08', 'certificates/collision_core_witnesses.jsonl.gz': 'ae64a5283400c61e15f4a8758d949f3ad8428645cb3691e0b5cdb539161f8a45'}}
CLAIM_MAP = {'source_commit': '838bca514679b2531d31f4e0dfd66641269e2a8c', 'mathematical_baseline_commit': 'd90313a717994936f254990f88d2624bbcce5bcd', 'scope': 'Navigation only, not evidence. Mathematical statements and evidence are verbatim ledger cells; review-only row M8 is omitted. Paper links locate annotations. Axiom records are supplied, not freshly built.', 'omitted_ledger_rows': ['M8'], 'ledger_sha256': '1eaf295bf512f6d56895cda4e6ebe383e512d6e136740346ad126840d8918f4f', 'ledger_frozen_prefix_sha256': '5ed828138081510847f5a89c53c152607138580d1cb041d4486904ccb63f49e9', 'axiom_log_sha256': '89d37d4717c1913c215d7923ede61ef7f3618c64f06f1f07dbda718a8d6418ae', 'tex_sha256': {'paper/tex/appA_data.tex': '0b1b94d1d4870a73f36727f67f6980093995ca0d1fd75975d8bbc8699c0f777e', 'paper/tex/appB_census.tex': '9bf45e4bc02575fbd1c9503ac399ad0ea2e7744152e320aa0c89ad8284534afa', 'paper/tex/appC_lean.tex': '4b4ecc732e1da0ef36fd0e343a28314a65413c1c99e87555bcc2da5b900b2e4b', 'paper/tex/appD_calibration.tex': 'c307ec368b5ea18f3886a8fa94923a86541e63d955219281c57ec87d47f97126', 'paper/tex/fig_collision_box.tex': 'fa5c4363c6e30d53a95687c0c1305651d10720a2eb9718ee3a94b74d47eaca6e', 'paper/tex/fig_companion_sectors.tex': 'd8f5d038c21e17e10caae5a1fbd05b5c48121ee274356a85b1a4467c4a57eba0', 'paper/tex/fig_feature_geometry.tex': 'a0744e9b03519b0c79b09fec4b895e32c7ad094ba33a4601f3787f109bcc5d93', 'paper/tex/fig_roadmap.tex': '6a9c10d92638ac45f6842861198578aa8c839a265696dab8f02d26057654de6e', 'paper/tex/generated/appendix_data_table.tex': '2c64c3efb52da75bdd32ce0adde2d53f8be1c75b83b7b3215ccbdf52f7b865ad', 'paper/tex/generated/atlas_44_table.tex': 'bb1ee7d0306b3bdf6814806ac7d76fc30982f5f5734be3edb01f09e9d221e9e1', 'paper/tex/generated/axioms_table.tex': '4fb3ed58193e0e78a7432c81729efd8879e2ded5320fb5e4c8e53469aa9ee8f9', 'paper/tex/generated/census_table.tex': '865ee13cfe4a51577deb8fa7cb40cf39bf647748454e5bcefe77ceb7a35d1588', 'paper/tex/generated/fig_carrier_panels.tex': '197042ea36c2e25fe3dfa27595d1196e14c2c559132db78320ee67c7ba7fe7ff', 'paper/tex/generated/fig_coincidence.tex': '3074a8fba6298e12f5071aebcde6c8b72f1902877fb5a9e8276903effd6d0c7f', 'paper/tex/generated/fig_nesting_slice.tex': 'c577ef09dd9d527902858eb739f142530e34470a0973b2ed19a31394cfa7bb92', 'paper/tex/generated/finite_theorems_table.tex': '0dcf9d94d49a3797eb4458a62bb6dc1ab96157f0109f834930ba2b555d728b1c', 'paper/tex/generated/first_shells_table.tex': '2e0b4792e4b695c1d19dbebbbd3244fc2cbb1e1e6d99c87d4f6f5c17203be88f', 'paper/tex/generated/hypotheses_record.tex': '11282eb1d419edede0bdcefee4b5eca2b5b3b2573095512935471540c1186815', 'paper/tex/generated/hypotheses_table.tex': 'c6fd5c08dc946a68c56571b6003c1517f0fe94c3dccc6a13b32eabb216e2cf7d', 'paper/tex/generated/mesh_table.tex': '86e1e2c397ba557911c29a1ad88e3bea4e6799c0a30527b46dfd40f30e421939', 'paper/tex/generated/parameter_family_table.tex': '570016fa5bc586f08286d49cf224281902182ac81cb393a6afb2768ac7f36292', 'paper/tex/macros.tex': '683d95e7e701ccd9b5489c6957d09f43c97861829f31154436f426124e3cd14e', 'paper/tex/main.tex': '7564c49ca424c2a4820ee5d2bbdb2d3563ee30702923935c944c7d430030b6c0', 'paper/tex/notation.tex': 'f1750396e2654ed3c0b2bc8c6f9b153e9f64ca6903c12b5fcb098fc8caebfcf6', 'paper/tex/sec1_intro.tex': '2d5c213ba218e2bc7385373b5e73005f604fe23d0790cb3fec172ff84d2f68b0', 'paper/tex/sec2_solid.tex': '32efc1d3c36c1e6b8d18b367ea7b19332f37c2583ea71febaab70695c7e7b2dc', 'paper/tex/sec3_finding.tex': 'ce09115f29da621dbfb7308669f1c680354c5b5c04a4c5f41ae434d4ce3b7ce8', 'paper/tex/sec4_companions.tex': '66f6fd398e425f2ee2ba216ec8927b37afd4605e6818cfff3df559bbe6e8eee0', 'paper/tex/sec5_registration.tex': '6b42fe6414d6b8a67c7e939ff3558b1a6c6a09b4151f204ccf1b9b00b182967e', 'paper/tex/sec6_hierarchy.tex': '0d527f42479ab2a4289ef315acd852bbce9bd1d2bd10813c4e44b323fa4331fb', 'paper/tex/sec7_aperiodicity.tex': '2a456ac67d11f7129ee19bb9426b63d429f8f0ef046955512cc8fccd036314aa', 'paper/tex/sec8_mechanization.tex': '2bd58d2d5dcbedb6d8894c9c71167cb3e058819c653861280f1a2e2e37cec9b9', 'paper/tex/sec9_remarks.tex': 'dad8cdb06bb47997a6cdce91c0b488e4cfd8d52fdf8a28ba2f5c0b3038b4d45b'}, 'rows': [{'id': 'Q1', 'statement': 'Socolar–Taylor pose the target: "a single, simply connected 2D or 3D prototile that forces maximal nonperiodicity by shape alone, or one that does not permit any weakly nonperiodic tilings"', 'ledger_tier': 'Q', 'evidence': '`1009.1419 L88`', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:18', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:5'], 'declarations': []}, {'id': 'Q2', 'statement': 'Kaplan 2025: SCD "cautionary tale"; "demand a strongly aperiodic 3D monotile, one that admits tilings whose symmetries never include an infinite cyclic subgroup of any kind"; "personally I am daunted by the prospect"', 'ledger_tier': 'Q', 'evidence': '`2509.12216 L246–L254`', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:19', 'paper_locations': ['paper/tex/sec1_intro.tex:5', 'paper/tex/sec1_intro.tex:63'], 'declarations': []}, {'id': 'Q3', 'statement': 'The hat\'s definitions of weakly/strongly periodic and aperiodic; "Following Mozes"', 'ledger_tier': 'Q', 'evidence': '`2303.10798 L34, L64`; Mozes Invent. Math. 128 (1997)', 'paper_sections': '1.5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:20', 'paper_locations': ['paper/tex/sec1_intro.tex:80', 'paper/tex/sec1_intro.tex:129', 'paper/tex/sec1_intro.tex:299'], 'declarations': []}, {'id': 'Q4', 'statement': 'In the plane weak and strong aperiodicity coincide for normal tiles (GS16 Thm 3.7.1)', 'ledger_tier': 'Q', 'evidence': '`2303.10798 L34`', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:21', 'paper_locations': ['paper/tex/sec1_intro.tex:80'], 'declarations': []}, {'id': 'Q5', 'statement': 'Coulbois et al.: weakly/mildly/strongly aperiodic by stabilizer; "strongly aperiodic monotiles are unknown in all settings"; the hat is mildly aperiodic', 'ledger_tier': 'Q', 'evidence': '`2409.15880 L43–L45, L88, L234, L236`', 'paper_sections': '1.1, 1.5, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:22', 'paper_locations': ['paper/tex/sec1_intro.tex:80', 'paper/tex/sec1_intro.tex:299', 'paper/tex/sec9_remarks.tex:3'], 'declarations': []}, {'id': 'Q6', 'statement': 'Socolar–Taylor\'s 3D tile: enforces R1/R2 by shape, genus zero after shifting plugs, "corrugated slabs … stack periodically"; "weakly nonperiodic tiling by Goodman-Strauss\'s definition"', 'ledger_tier': 'Q', 'evidence': '`1003.4279 L171`; `1009.1419 L69–L71`', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:23', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:63'], 'declarations': []}, {'id': 'Q7', 'statement': 'SCD tile, named "Schmitt–Conway–Danzer biprism" as the hat paper and Kaplan name it: a rhombic biprism, convex (decorating it with chiral plugs "loses the appealing property of convexity"), twisted layers, screw symmetry, reflections allow a periodic tiling, "heterogeneously periodic"; history: Schmitt\'s unpublished 1988 deformed cube and Conway\'s smoothing (Radin, who names neither Danzer nor Senechal nor convexity); details cited through Senechal §7.2 as the hat paper does (Senechal\'s text not in the corpus)', 'ledger_tier': 'Q', 'evidence': '`2303.10798 L33`; `2509.12216 L246–L249`; `1009.1419 L14–L16, L23, L30, L72`; `2008.09085` (dossier kaplan_radin_maiti.md); Sen96 §7.2 via SMKGS24', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:24', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:63', 'paper/tex/sec1_intro.tex:139'], 'declarations': []}, {'id': 'Q8', 'statement': 'Greenfeld–Tao: translational, "sufficiently large d", "extremely large", Z³ open, tile "need not be connected"; they file the hat separately; Greenfeld–Kolountzakis: the tile can be connected', 'ledger_tier': 'Q', 'evidence': '`2211.15847 L5, L24–L28, L34, L600–L603`; Annals 200 (2024) 301–363; `2303.10798 L35` (GK23 connected)', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:25', 'paper_locations': ['paper/tex/sec1_intro.tex:41'], 'declarations': []}, {'id': 'Q9', 'statement': 'Hat: closed topological disk, needs reflections; two proofs; 188 patches cross-checked by two implementations; fault lines open', 'ledger_tier': 'Q', 'evidence': '`2303.10798 L13, L51–L54, L158, L253`', 'paper_sections': '1.1, 1.4, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:26', 'paper_locations': ['paper/tex/sec1_intro.tex:56'], 'declarations': []}, {'id': 'Q10', 'statement': 'Spectre: definitions of homochiral tiling and strictly chiral aperiodic monotile; unique hierarchy ⇒ non-periodic (GS16 Thm 10.1.1)', 'ledger_tier': 'Q', 'evidence': '`2305.17743 L22, L43, L45`', 'paper_sections': '1.5, 7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:27', 'paper_locations': ['paper/tex/sec1_intro.tex:56', 'paper/tex/sec1_intro.tex:318'], 'declarations': []}, {'id': 'Q11', 'statement': 'Goodman-Strauss: unique decomposition is the load-bearing property; existence by compactness', 'ledger_tier': 'Q', 'evidence': '`1608.07165 L80, L308`', 'paper_sections': '1.3, 6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:28', 'paper_locations': ['paper/tex/sec1_intro.tex:191'], 'declarations': ['R44.existence']}, {'id': 'Q12', 'statement': 'The near-misses and what each concedes, in the hat\'s words: Gummelt overlaps; Penrose 1+ε+ε² ("no matter how thin or small they become, the other tiles remain necessary"); Taylor–Socolar markings, or a disconnected tile / cutpoints / a 3D shape tiling a thickened plane; Walton–Whittaker orientational rules; Mampusti–Whittaker: "not an einstein in the technical sense"', 'ledger_tier': 'Q', 'evidence': '`2303.10798 L28–L32`; `1903.01158 L11`', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:29', 'paper_locations': ['paper/tex/sec1_intro.tex:41'], 'declarations': []}, {'id': 'Q13', 'statement': "Fletcher: one cubic prototile tiles R³ aperiodically with a 1-corona atlas rule, MLD to Kari's Wang cubes", 'ledger_tier': 'Q', 'evidence': '`1003.4909 L45–L48`', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:30', 'paper_locations': ['paper/tex/sec1_intro.tex:63'], 'declarations': []}, {'id': 'Q14', 'statement': 'Jeandel–Rao / Labbé computation-section forms ("a kind of certificate"; "impossible without a computer"; "Code." paragraph)', 'ledger_tier': 'Q', 'evidence': '`1506.06492 L94`; `1808.07768 L51, L253`', 'paper_sections': '1.4, 8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:31', 'paper_locations': ['paper/tex/sec1_intro.tex:255'], 'declarations': []}, {'id': 'Q15', 'statement': 'Socolar 2023 contrast: hat quasicrystalline vs Taylor–Socolar limit-periodic with peaks 2^{-n}k₀', 'ledger_tier': 'Q', 'evidence': '`2305.01174 L137`', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:32', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'Q16', 'statement': 'Lee–Moody 2001 Theorem 3, four-way equivalence and hypotheses', 'ledger_tier': 'Q/C', 'evidence': '`math0002019 L123–L127`; Discrete Comput. Geom. 25 (2001)', 'paper_sections': '8, App.', 'ledger_location': 'paper/CLAIMS_LEDGER.md:33', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:179', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'Q17', 'statement': "Schlottmann's theorem (regular model set ⇒ pure point)", 'ledger_tier': 'C', 'evidence': '`math0002019 L108`; `0910.4450 L392–L397` (LMS03 Thm 5.11/5.12)', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:34', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'Q18', 'statement': '"limit-periodic" as a term (countably, not finitely, generated Fourier module; p-adic internal space)', 'ledger_tier': 'Q', 'evidence': '`math-ph9901008 L10, L121`; `1007.0707 L3, L49–L50`', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:35', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'Q19', 'statement': 'Akiyama–Lee: 168-prototile Taylor–Socolar substitution, overlap coincidence, cost disclosed', 'ledger_tier': 'Q', 'evidence': '`1212.4209 L48, L62–L64`', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:36', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'Q20', 'statement': "Myers's Lean staging repository for the hat/Spectre (in progress, not mathlib)", 'ledger_tier': 'C', 'evidence': 'github.com/jsm28/AperiodicMonotilesLean (web scan)', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:37', 'paper_locations': ['paper/tex/sec1_intro.tex:255'], 'declarations': []}, {'id': 'Q21', 'statement': "History in the hat's order: Wang 1961, Berger 1966 (20,426 tiles), Robinson 1971 (six tiles), Penrose 1978 (two tiles), Jeandel–Rao (11 Wang tiles is the minimum for Wang tiles; the six/two counts use other motions and conventions)", 'ledger_tier': 'Q', 'evidence': '`2303.10798 L18–L24, L36, L253` (Robinson "six shapes"); `hat_references.txt` entries Wan61, Ber66, Rob71, Pen78, JR21', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:38', 'paper_locations': ['paper/tex/sec1_intro.tex:31', 'paper/tex/sec1_intro.tex:41'], 'declarations': []}, {'id': 'Q22', 'statement': 'Reader-engagement sentence pointing to the interactive viewer', 'ledger_tier': 'Q', 'evidence': '`2303.10798 L96`; `viewer/index.html`', 'paper_sections': '3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:39', 'paper_locations': ['paper/tex/appA_data.tex:3', 'paper/tex/appA_data.tex:86', 'paper/tex/sec3_finding.tex:55'], 'declarations': []}, {'id': 'Q23', 'statement': 'The problem is posed by Socolar–Taylor (Q1) and described by Kaplan as demanded (Q2); the paper does NOT call it "longstanding" or "open" in anyone\'s words but theirs', 'ledger_tier': 'Q', 'evidence': 'Q1, Q2; dossier `kaplan_radin_maiti.md` (Kaplan never says "open")', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:40', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:5'], 'declarations': []}, {'id': 'Q24', 'statement': "Hilbert's 18th problem, second part, asked whether anisohedral polyhedra exist in R³; Reinhardt found one; Heesch gave a planar example", 'ledger_tier': 'Q', 'evidence': '`2303.10798 L37–L39`', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:41', 'paper_locations': ['paper/tex/sec1_intro.tex:63'], 'declarations': []}, {'id': 'Q25', 'statement': 'After the hat and Spectre: simplified proofs (Akiyama–Araki) and the structure of the tilings (Baake–Gähler–Sadun; Baake et al. diffraction; Socolar), one sentence', 'ledger_tier': 'C', 'evidence': "the papers' own abstracts: `2307.12322` (AA25), `2305.05639 L1–L14` (BGS25), `2502.03268` (BGMM25), `2305.01174` (Soc23); dossiers", 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:42', 'paper_locations': ['paper/tex/sec1_intro.tex:56'], 'declarations': []}, {'id': 'Q26', 'statement': "In the hyperbolic plane weak aperiodicity comes readily: Böröczky's weakly aperiodic monotile (1974); Goodman-Strauss's strongly aperiodic set (2005)", 'ledger_tier': 'Q/C', 'evidence': '`2303.10798 L33` (Böröczky, "appear readily in the hyperbolic plane"); GS05 title/abstract (`GS05_hyperbolic.lines`)', 'paper_sections': '1.1', 'ledger_location': 'paper/CLAIMS_LEDGER.md:43', 'paper_locations': ['paper/tex/sec1_intro.tex:80'], 'declarations': []}, {'id': 'T1', 'statement': 'Q admits a tiling of R³', 'ledger_tier': 'T1n (unconditional)', 'evidence': '`r44_einstein`, first conjunct `Nonempty (Tiling Q)` (`R44/LogicalSpine.lean`)', 'paper_sections': '1.2, 3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:49', 'paper_locations': ['paper/tex/appC_lean.tex:3', 'paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:5', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec3_finding.tex:200'], 'declarations': ['R44.existence', 'R44.r44_einstein', 'R44.small_collar_realization_holds']}, {'id': 'T2', 'statement': 'For every tiling T, Per(T) = {0}', 'ledger_tier': 'T1n (unconditional)', 'evidence': '`r44_einstein`, `Per T = {0}`', 'paper_sections': '1.2, 7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:50', 'paper_locations': ['paper/tex/appC_lean.tex:3', 'paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec7_aperiodicity.tex:3'], 'declarations': ['R44.no_period', 'R44.period_halving', 'R44.r44_einstein', 'R44.sym_card_le_24']}, {'id': 'T3', 'statement': 'For every tiling T, \\|Sym(T)\\| ≤ 24', 'ledger_tier': 'T1n (unconditional)', 'evidence': '`r44_einstein`, `encard ≤ 24`', 'paper_sections': '1.2, 7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:51', 'paper_locations': ['paper/tex/appC_lean.tex:3', 'paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:5', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec7_aperiodicity.tex:3'], 'declarations': ['R44.no_period', 'R44.period_halving', 'R44.r44_einstein', 'R44.sym_card_le_24']}, {'id': 'T4', 'statement': 'No tiling has a symmetry of infinite order; hence Q is strongly aperiodic (Mozes/hat sense)', 'ledger_tier': 'derived', 'evidence': 'T1 (existence) + T3 (every tiling has finite symmetry group) + a finite group has no infinite-order element; definition Q3', 'paper_sections': '1.2, 7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:52', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:80', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec7_aperiodicity.tex:3'], 'declarations': ['R44.existence', 'R44.no_period', 'R44.period_halving', 'R44.r44_einstein', 'R44.sym_card_le_24']}, {'id': 'T5', 'statement': 'In the taxonomy of CGGL24, Q is mildly aperiodic; trivial stabilizers not claimed', 'ledger_tier': 'derived', 'evidence': 'T1 (existence) + T3 (every tiling has finite symmetry group) + Q5 (definition: mildly aperiodic = every cotiler has finite stabilizer; Q has no self-symmetry, O5, so stabilizer = symmetry group); N3', 'paper_sections': '1.1, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:53', 'paper_locations': ['paper/tex/sec1_intro.tex:80', 'paper/tex/sec1_intro.tex:299', 'paper/tex/sec7_aperiodicity.tex:3'], 'declarations': ['R44.existence', 'R44.no_period', 'R44.period_halving', 'R44.r44_einstein', 'R44.sym_card_le_24']}, {'id': 'T6', 'statement': 'Every tiling is homochiral; all placements +1 or all −1; no mixed pair', 'ledger_tier': 'T1n (unconditional)', 'evidence': '`tiling_homochiral`, `tiling_chirality_corollary`', 'paper_sections': '7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:54', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec1_intro.tex:139', 'paper/tex/sec7_aperiodicity.tex:56'], 'declarations': ['R44.tiling_chirality_corollary', 'R44.tiling_homochiral']}, {'id': 'T7', 'statement': 'Q is a strictly chiral aperiodic monotile of R³ (Spectre definition)', 'ledger_tier': 'derived', 'evidence': 'T1 (existence) + T2 (every tiling non-periodic) + T6 (every tiling homochiral) + Q10 (definition: a tile that admits only homochiral non-periodic tilings); an all-orientation-reversing tiling is homochiral since any two copies are related by the orientation-preserving relative motion h g⁻¹', 'paper_sections': '7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:55', 'paper_locations': ['paper/tex/sec1_intro.tex:139', 'paper/tex/sec7_aperiodicity.tex:56'], 'declarations': ['R44.existence', 'R44.tiling_chirality_corollary', 'R44.tiling_homochiral']}, {'id': 'T8', 'statement': 'Each tiling carries, at every level n, a unique partition of the registered cells into 2ⁿ-scaled chairs in registered poses, nested through the eight child poses; level 0 is the carrier tiling', 'ledger_tier': 'T1n (unconditional), for arbitrary geometrically nested registered families', 'evidence': '`carrier_hierarchy`, `carrier_hierarchy_unique`, `geometric_hierarchy_canonical`, `geometric_hierarchy_unique`, `carrierHierarchy_geometric` (LogicalSpine.lean at the pin; M9 closed)', 'paper_sections': '6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:56', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec1_intro.tex:154', 'paper/tex/sec6_hierarchy.tex:3', 'paper/tex/sec6_hierarchy.tex:87', 'paper/tex/sec6_hierarchy.tex:108'], 'declarations': ['R44.carrierHierarchy_geometric', 'R44.carrier_hierarchy', 'R44.carrier_hierarchy_unique', 'R44.first_shells_33', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique']}, {'id': 'T9', 'statement': 'No exhaustion clause; non-exhausting hierarchies occur (L10) and are not thereby excluded from the substitution hull; hull membership of every tiling open', 'ledger_tier': 'N / T2', 'evidence': 'ruling R7, MECHANIZATION_PLAN.md C3; L10', 'paper_sections': '6, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:57', 'paper_locations': ['paper/tex/sec1_intro.tex:154', 'paper/tex/sec6_hierarchy.tex:3', 'paper/tex/sec6_hierarchy.tex:87', 'paper/tex/sec6_hierarchy.tex:123', 'paper/tex/sec8_mechanization.tex:123'], 'declarations': ['R44.carrier_hierarchy', 'R44.carrier_hierarchy_unique', 'R44.first_shells_33', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique', 'R44.seed_language_closure']}, {'id': 'T10', 'statement': 'Reflections allowed; no face-to-face, lattice, common-orientation, connected-contact-graph or local-finiteness assumption', 'ledger_tier': 'D', 'evidence': '`Tiling Q` over `RigidMotion`; THEOREM.md statement', 'paper_sections': '1.2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:58', 'paper_locations': ['paper/tex/sec1_intro.tex:98', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec1_intro.tex:129'], 'declarations': []}, {'id': 'T11', 'statement': 'Local finiteness is proved, not assumed', 'ledger_tier': 'T1', 'evidence': '`local_finiteness` (`R44/LogicalSpineFoundation.lean`), A-L2.1', 'paper_sections': '1.5, 4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:59', 'paper_locations': ['paper/tex/sec1_intro.tex:129'], 'declarations': ['R44.local_finiteness']}, {'id': 'T12', 'statement': 'The `Hypotheses` record, whose fields quoted the written lemmas one by one, is empty at the pin: all 19 former fields are theorems in `lean/R44/R44/Proved/`; `r44_einstein` takes no hypothesis and the conditional form survives as `r44_einstein_of_hypotheses` over the empty record; the theorem is kernel-checked modulo the 21 named compiler hooks; the written proofs remain as exposition', 'ledger_tier': 'D', 'evidence': '`lean/R44/HYPOTHESES.md` (Discharged table; Remaining fields: empty) and `LogicalSpine.lean` at d90313a717; `lean/R44/PAPER_UPDATES.md` §1–§3', 'paper_sections': '1.4, 8, App. C', 'ledger_location': 'paper/CLAIMS_LEDGER.md:60', 'paper_locations': ['paper/tex/appC_lean.tex:3', 'paper/tex/appC_lean.tex:20', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec1_intro.tex:164', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec8_mechanization.tex:5', 'paper/tex/sec8_mechanization.tex:17', 'paper/tex/sec8_mechanization.tex:69'], 'declarations': ['R44.carrier_hierarchy', 'R44.geometric_hierarchy_unique', 'R44.r44_einstein', 'R44.r44_einstein_of_hypotheses', 'R44.tiling_homochiral']}, {'id': 'T13', 'statement': 'Period halving p/2ⁿ ∈ Z³ ⇒ p = 0; the 24-frame injection; proved outright in Lean', 'ledger_tier': 'T1 for `no_period` and the conditional `sym_card_le_24`; T1n for period halving and their application to Q', 'evidence': 'THEOREM.md "Lean spine" paragraph; `no_period`, `all_periods_grid`', 'paper_sections': '7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:61', 'paper_locations': ['paper/tex/sec1_intro.tex:232', 'paper/tex/sec7_aperiodicity.tex:3'], 'declarations': ['R44.no_period', 'R44.period_halving', 'R44.r44_einstein', 'R44.sym_card_le_24']}, {'id': 'T14', 'statement': 'Parameter family: heights c_j/10000 with the R§11 conditions; proof unchanged; no perturbation stability claimed', 'ledger_tier': 'T3', 'evidence': 'R§11', 'paper_sections': '9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:62', 'paper_locations': ['paper/tex/sec2_solid.tex:136'], 'declarations': []}, {'id': 'O1', 'statement': 'Q = seven-cube chair (2×2×2 minus a corner) with 24 exposed unit panels, 8 pyramids per panel, heights ±j/10000, j=1..12, base half-width 1/100; rational coordinates; volume 7', 'ledger_tier': 'T1n', 'evidence': '`solid_mesh_exact`; `solid/r44_solid.json` sha256 f320d7a0…', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:68', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/notation.tex:3', 'paper/tex/sec1_intro.tex:5', 'paper/tex/sec1_intro.tex:98', 'paper/tex/sec2_solid.tex:3', 'paper/tex/sec9_remarks.tex:44'], 'declarations': ['R44.solid_mesh_exact']}, {'id': 'O2', 'statement': '2,138 vertices, 6,408 edges, 4,272 triangles; V − E + F = 2', 'ledger_tier': 'T1n', 'evidence': '`boundary_sphere` (`R44/Theorems.lean`)', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:69', 'paper_locations': ['paper/tex/sec2_solid.tex:69'], 'declarations': ['R44.boundary_sphere', 'R44.mesh_angle_audit']}, {'id': 'O3', 'statement': 'Boundary is a closed connected triangulated surface; every edge in two triangles; single-cycle vertex links', 'ledger_tier': 'T1n / T2', 'evidence': '`boundary_sphere`; `verify/boundary_sphere.py`', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:70', 'paper_locations': ['paper/tex/sec2_solid.tex:69'], 'declarations': ['R44.boundary_sphere', 'R44.mesh_angle_audit']}, {'id': 'O4', 'statement': 'Q is a closed topological 3-ball', 'ledger_tier': 'C', 'evidence': 'O3 + classification of closed surfaces + PL Schoenflies (Moise; Rourke–Sanderson), cited T3', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:71', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:5', 'paper/tex/sec1_intro.tex:98', 'paper/tex/sec2_solid.tex:69'], 'declarations': ['R44.boundary_sphere', 'R44.mesh_angle_audit']}, {'id': 'O5', 'statement': 'Q has no self-isometry', 'ledger_tier': 'T1n (components T1 / T1n / T1 at the pinned commit)', 'evidence': 'reduction of an arbitrary self-isometry to a carrier-preserving motion: `planar_area_carrier_recovery_holds` (T1, formal proof by diameter endpoints; the planar-area argument is the written proof, ERRATA E5); carrier-preserving motion to a feature-table frame: `carrier_feature_frame_reduction_holds` (T1n, hooks `profile_canonical`, `transported_role_geometry`); the 48-frame comparison: `no_native_symmetry` (T1); assembly `native_asymmetry_reduction`', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:72', 'paper_locations': ['paper/tex/sec1_intro.tex:176', 'paper/tex/sec1_intro.tex:299', 'paper/tex/sec2_solid.tex:92'], 'declarations': ['R44.carrier_feature_frame_reduction_holds', 'R44.native_asymmetry_reduction', 'R44.no_native_symmetry', 'R44.planar_area_carrier_recovery_holds', 'R44.profile_canonical']}, {'id': 'O6', 'statement': '192 feature roles with distinct eighth-grid centres; 24 deviations distinct (10000 i² = 20000 j² + j⁴ unsolvable)', 'ledger_tier': 'T1', 'evidence': '`native_features_length`, `native_feature_centers_nodup`, `deviations_distinct`', 'paper_sections': '2, 4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:73', 'paper_locations': ['paper/tex/sec2_solid.tex:3', 'paper/tex/sec2_solid.tex:115', 'paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:60'], 'declarations': ['R44.complete_dihedral_list_holds', 'R44.deviations_distinct', 'R44.local_finiteness', 'R44.mesh_angle_audit', 'R44.native_feature_centers_nodup', 'R44.native_features_length']}, {'id': 'O7', 'statement': 'Every mesh edge has a declared dihedral; 1,536 feature edges, 32 per class', 'ledger_tier': 'T1n', 'evidence': '`mesh_angle_audit`', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:74', 'paper_locations': ['paper/tex/sec2_solid.tex:115'], 'declarations': ['R44.mesh_angle_audit']}, {'id': 'O8', 'statement': 'Eight child poses partition 2P as integer cell sets', 'ledger_tier': 'T1', 'evidence': '`children_partition_2P`', 'paper_sections': '3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:75', 'paper_locations': ['paper/tex/sec1_intro.tex:232', 'paper/tex/sec3_finding.tex:5'], 'declarations': ['R44.children_partition_2P']}, {'id': 'O9', 'statement': 'Volume 7', 'ledger_tier': 'T1n', 'evidence': '`solid_mesh_exact`', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:76', 'paper_locations': ['paper/tex/sec2_solid.tex:3'], 'declarations': ['R44.solid_mesh_exact']}, {'id': 'O10', 'statement': 'Q is not convex', 'ledger_tier': 'T2', 'evidence': '`paper/scripts/nonconvex_witness.py` (stdlib, exact rationals): the mesh vertices u = (2,2,1) and v = (2,1,2) are in Q (Q is closed and contains its boundary mesh), their midpoint m = (2,3/2,3/2) is outside Q by an exact ray-parity test along the generic direction (1,1/7,1/11) with 0 proper crossings and 0 boundary hits, after an exact check that m lies on no closed triangle of the surface (0 of 4,272), controls (1/2,1/2,1/2) inside and (3,3,3) outside; output `STATUS: PASS`', 'paper_sections': '1.1, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:77', 'paper_locations': ['paper/tex/sec1_intro.tex:98'], 'declarations': []}, {'id': 'O11', 'statement': 'Planar boundary areas of Q on the nine coordinate planes: 2492/625 at height 0, 623/625 at height 1, 1869/625 at height 2 (each axis), each > 9/10; all non-axis facets together < 1212/15625 < 1/4; the three area types identify the height-0, height-1 and height-2 plane triples, so a self-isometry fixes the origin, the corner (2,2,2) and the notch, hence P', 'ledger_tier': 'T2 (areas) / T3 (the written lemma `planar_area_carrier_recovery`)', 'evidence': '`paper/scripts/planar_areas.py` (stdlib, exact rationals from `solid/r44_solid.json`; `STATUS: PASS`); external review 2026-09-09 m2', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:78', 'paper_locations': ['paper/tex/sec2_solid.tex:92'], 'declarations': ['R44.planar_area_carrier_recovery_holds']}, {'id': 'O12', 'statement': 'Figures 1 and 3 are exaggerated illustrations of Q, not true-scale renderings: the seven-cube carrier and all 192 feature centres at their canonical coordinates, feature bases widened ×5 about their centres, signed heights ×80, polarities and relative heights preserved, no feature added, omitted or moved; panel 13 (outward normal −y, centre (3/2, 0, 3/2), global tangent axes x, z) carries roles 104–111 with signed coefficients −9, −10, −11, −12, +9, +11, +10, +12 (true height a/10000, true base side 1/50); the sections show its +9 and −9 features', 'ledger_tier': 'D', 'evidence': '`figures/print-preview/build_preview.py` (192/192 feature records checked against the mesh and `solid/native_panels.csv`, rational arithmetic), `audit_preview.py` (768 display triangles read back), `feature-audit.csv` rows 104–111, `visibility-audit.csv`, `SHA256SUMS`; owner-supplied artwork 2026-09-09 replacing the `render_r44.py` renders in Figures 1 and 3', 'paper_sections': '1, 2, A', 'ledger_location': 'paper/CLAIMS_LEDGER.md:79', 'paper_locations': ['paper/tex/appA_data.tex:86', 'paper/tex/sec1_intro.tex:5', 'paper/tex/sec2_solid.tex:3'], 'declarations': []}, {'id': 'D1', 'statement': 'Tile, tiling, admits, monohedral (congruence includes reflections), locally finite', 'ledger_tier': 'D', 'evidence': 'Grünbaum–Shephard via `2303.10798 L57–L58`, with "closed topological disk" replaced by "closed topological 3-ball"', 'paper_sections': '1.5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:85', 'paper_locations': ['paper/tex/notation.tex:3', 'paper/tex/sec1_intro.tex:129', 'paper/tex/sec1_intro.tex:281', 'paper/tex/sec1_intro.tex:299'], 'declarations': []}, {'id': 'D2', 'statement': 'A monotile is a tile admitting a monohedral tiling; an aperiodic monotile or einstein here is a closed topological 3-ball admitting tilings but only non-periodic ones, by geometry alone, with no non-geometric matching rule', 'ledger_tier': 'D', 'evidence': '`2303.10798 L25` restated for 3-balls', 'paper_sections': '1.5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:86', 'paper_locations': ['paper/tex/sec1_intro.tex:281'], 'declarations': []}, {'id': 'D3', 'statement': 'Registered, registration, the atlas, features, panels, carrier, supertile of level n', 'ledger_tier': 'D', 'evidence': 'our definitions in §2–§3 from `solid/r44_solid.json`, `certificates/candidate_certificate.json`, `LogicalSpine.lean` (`Registration`, `RegisteredPose`, `CarrierHierarchy`)', 'paper_sections': '2, 3, 6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:87', 'paper_locations': ['paper/tex/notation.tex:3', 'paper/tex/sec1_intro.tex:281', 'paper/tex/sec3_finding.tex:5', 'paper/tex/sec3_finding.tex:55'], 'declarations': []}, {'id': 'D4', 'statement': 'Public qualifier: "proof submission" until the written lemmas are formalized or refereed', 'ledger_tier': 'D', 'evidence': '`README.md` L9–L12 and the review-status paragraph; HANDOFF.md §5', 'paper_sections': '8, front matter', 'ledger_location': 'paper/CLAIMS_LEDGER.md:88', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/main.tex:67', 'paper/tex/sec1_intro.tex:107', 'paper/tex/sec1_intro.tex:164', 'paper/tex/sec8_mechanization.tex:5', 'paper/tex/sec8_mechanization.tex:17', 'paper/tex/sec8_mechanization.tex:69'], 'declarations': ['R44.carrier_hierarchy', 'R44.geometric_hierarchy_unique', 'R44.r44_einstein', 'R44.r44_einstein_of_hypotheses', 'R44.tiling_homochiral']}, {'id': 'D5', 'statement': 'Homochiral tiling; strictly chiral aperiodic monotile', 'ledger_tier': 'D', 'evidence': '`2305.17743 L22` verbatim', 'paper_sections': '1.5, 7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:89', 'paper_locations': ['paper/tex/notation.tex:3', 'paper/tex/sec1_intro.tex:139', 'paper/tex/sec1_intro.tex:318', 'paper/tex/sec7_aperiodicity.tex:56'], 'declarations': ['R44.tiling_chirality_corollary', 'R44.tiling_homochiral']}, {'id': 'D6', 'statement': 'Apparatus: keywords "Aperiodic monotile, einstein problem, strongly aperiodic tilings, tilings of three-dimensional space, polyhedral tiles, substitution tilings, hierarchical tilings, limit-periodic model sets, computer-assisted proof, Lean 4 formalization, Six Birds Theory, emergence calculus" (owner, 2026-09-09); MSC 05B45, 52C22, 52C23; CC BY; DOI/arXiv/URL on every reference; "Code." paragraph', 'ledger_tier': 'D', 'evidence': 'COMMUNITY_PAPER_PROFILE.md §2 item 7, §7', 'paper_sections': 'front/back matter', 'ledger_location': 'paper/CLAIMS_LEDGER.md:90', 'paper_locations': ['paper/tex/main.tex:56'], 'declarations': []}, {'id': 'D7', 'statement': 'The chirality/congruence convention paragraph, then that Q needs no such caveat', 'ledger_tier': 'Q/derived', 'evidence': '`2303.10798 L52–L54`; T6', 'paper_sections': '1.2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:91', 'paper_locations': ['paper/tex/sec1_intro.tex:139'], 'declarations': []}, {'id': 'D8', 'statement': 'The planar calibration (Appendix D): the hat reconstructed as an exact polygon with its published substitution and recognition grammar replayed as imports; an abstract Lean theorem (`CoarseningTower`, `period_lifts`, `tower_forces_trivial_stabilizers`, `nonempty_tower_landing`) that transports a period unchanged through coarsening levels and excludes nonzero periods by unbounded lower bounds; a tooling calibration, not a result, and not the R44 halving normalization', 'ledger_tier': 'D', 'evidence': '`calibration_2d/LANDING.md`; `calibration_2d/rounds/r004_varying_geometry_tower/formal/Tower.lean`', 'paper_sections': 'App. D', 'ledger_location': 'paper/CLAIMS_LEDGER.md:92', 'paper_locations': ['paper/tex/appD_calibration.tex:3'], 'declarations': []}, {'id': 'D9', 'statement': "Patch: a finite subfamily of a packing, without the hat's topological-ball condition; 1-corona: the tiles touching a chosen tile; cut-and-project scheme, model set, regular model set, 2-adic internal group, modular coincidence as in Lee–Moody", 'ledger_tier': 'D', 'evidence': '`2303.10798 L61` (hat patch convention); `1003.4909 L25` (Fletcher 1-corona); `math0002019 L102–L107` (Lee–Moody definitions), `L31–L32` (modular coincidence), `L111–L115` (the completion and the cut-and-project construction)', 'paper_sections': '1.5, 8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:93', 'paper_locations': ['paper/tex/sec1_intro.tex:281', 'paper/tex/sec8_mechanization.tex:179'], 'declarations': []}, {'id': 'H1', 'statement': 'Periodicity as descent, at the symbolic level: a lattice labelling x : Z³ → A is p-periodic iff it factors through Z³/⟨p⟩ (elementary, one line). Bridge to tilings of Q, stated only as far as the pinned declarations go: every tiling admits a registration (`registration_of_tiling`: one ambient isometry after which every placement is one of the 24 proper cubic frames plus an integer shift; T1n through `unrestricted_alignment_holds` (A15), `component_solids_cover_holds` (A14), `baseline_component_covers_grid_holds` (A13)), and under registration plus native asymmetry (`NativeAsymmetric Q`, T1n: assembled by `native_asymmetry_reduction` from `planar_area_carrier_recovery_holds` (R§6 / E5, T1), the carrier-frame reduction `carrier_feature_frame_reduction_holds` (T1n) and the 48-frame comparison `no_native_symmetry` (T1)) every translational period of the tiling is an integer vector in the ambient frame (`period_grid_from_registered_poses`, which requires `NativeAsymmetric Q`). This is the one direction the proof uses; no packaged equivalence between geometric periods and the periods of a 168-label encoding is claimed or cited. An einstein is a rule with admissible global labellings none of which descends; for Q the conclusion Per(T) = {0} is derived directly, by period halving (`period_halving`) and integrality at every level (`r44_einstein`)', 'ledger_tier': 'elementary (symbolic part) + T1n bridge by named declarations', 'evidence': 'one-line proof; `registration_of_tiling`; `period_grid_from_registered_poses`; `period_halving`; Foundations VI L2010–L2020 for the phrasing', 'paper_sections': '1.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:99', 'paper_locations': ['paper/tex/sec1_intro.tex:176'], 'declarations': ['R44.baseline_component_covers_grid_holds', 'R44.carrier_feature_frame_reduction_holds', 'R44.component_solids_cover_holds', 'R44.native_asymmetry_reduction', 'R44.no_native_symmetry', 'R44.period_halving', 'R44.planar_area_carrier_recovery_holds', 'R44.r44_einstein', 'R44.registration_of_tiling', 'R44.unrestricted_alignment_holds']}, {'id': 'H2', 'statement': 'Theorem G11 verbatim (two hypotheses, conclusion); calibration list ending with the hat and Spectre; role reading P2/P4/P1', 'ledger_tier': 'Q', 'evidence': 'Foundations VI L2044–L2056, L2086–L2096, L2102–L2106; doi:10.5281/zenodo.22254289', 'paper_sections': '1.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:100', 'paper_locations': ['paper/tex/sec1_intro.tex:226', 'paper/tex/sec3_finding.tex:144'], 'declarations': ['R44.existence', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique']}, {'id': 'H3', 'statement': 'A hierarchy certifies aperiodicity only if forced: equivariant extraction preserves stabilizers; recognizability + unique composition + period halving', 'ledger_tier': 'Q / elementary', 'evidence': '`1608.07165 L80, L308`; `2305.17743 L43`; the stabilizer inclusion Stab(x) ⊆ Stab(q(x)) (one line, stated)', 'paper_sections': '1.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:101', 'paper_locations': ['paper/tex/sec1_intro.tex:191'], 'declarations': []}, {'id': 'H4', 'statement': 'One shape puts the admissibility rule into the carrier; three requirements for a marked system to become a bare solid (existence; every geometric realization decodes; a geometric period is a decoded period); "one shape ≠ one role"', 'ledger_tier': 'D', 'evidence': 'P3/P4/P5 `SOURCE_AUDIT.md` "SBT interpretation"; `history/cascade/findings.md` L44', 'paper_sections': '1.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:102', 'paper_locations': ['paper/tex/sec1_intro.tex:202', 'paper/tex/sec3_finding.tex:144'], 'declarations': ['R44.existence', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique']}, {'id': 'H5', 'statement': "The design test (a diagnostic, not a theorem): the halving argument needs the coarsened tiling to be again a legal tiling of Q, i.e. the decoded parent contact language must lie inside the fine language; equality of the two languages is the sufficient form that is checked, finite form for Q `parent_atlas_eq_fine`. Sufficient, not necessary (a strictly smaller decoded language would also preserve legality). When the decoded language is strictly larger the argument fails and whether a periodic parent tiling exists must be checked; in the one case that arose (P4) it existed, for that case's language only", 'ledger_tier': 'T1n (the identity) / D (the test)', 'evidence': '`parent_atlas_eq_fine`; R§8; P4 `PROOFS.md` L104–L139', 'paper_sections': '1.3, 3.2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:103', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:214', 'paper/tex/sec3_finding.tex:83'], 'declarations': ['R44.parent_atlas_eq_fine']}, {'id': 'H6', 'statement': 'The framework supplied the reading and the test, not the proofs; framework vocabulary certifies nothing; claims are instrument-indexed; no literalization', 'ledger_tier': 'D', 'evidence': 'non-descending-objects paper (dossier D §5: "This vocabulary alone does not certify any claim"; no-overreading theorem)', 'paper_sections': '1.3, 8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:104', 'paper_locations': ['paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:226', 'paper/tex/sec3_finding.tex:144', 'paper/tex/sec3_finding.tex:191'], 'declarations': ['R44.existence', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique']}, {'id': 'H7', 'statement': 'Step 1: the marked chair: a single-marking local rule admits the chair substitution and no periodic tiling up to L = 7 (SAT); the all-scale recursion is recorded as an honest gap; the marks could not be geometrized', 'ledger_tier': 'D (history, finite evidence only)', 'evidence': '`history/cascade/findings.md` L40–L48; `LINEAGE.md` item 1', 'paper_sections': '3.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:105', 'paper_locations': ['paper/tex/sec3_finding.tex:106', 'paper/tex/sec3_finding.tex:110'], 'declarations': []}, {'id': 'H8', 'statement': 'Step 2: first bare solid is a periodic control; 62 fine contacts decode to 398 parent contacts', 'ledger_tier': 'T2 (packet replay)', 'evidence': 'P3 `README.md`; P4 `PROOFS.md` L106, L136; P4 `README.md`', 'paper_sections': '3.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:106', 'paper_locations': ['paper/tex/sec3_finding.tex:106', 'paper/tex/sec3_finding.tex:119'], 'declarations': []}, {'id': 'H9', 'statement': 'Step 3: the frame change; 44 = 44 with the full 24-element frame group', 'ledger_tier': 'T1n', 'evidence': 'P5 `PROOFS.md` "Claim and evidence status"; `parent_atlas_eq_fine`, `orientation_group_24`', 'paper_sections': '3.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:107', 'paper_locations': ['paper/tex/sec3_finding.tex:106', 'paper/tex/sec3_finding.tex:132'], 'declarations': ['R44.orientation_group_24', 'R44.parent_atlas_eq_fine']}, {'id': 'H10', 'statement': 'The method proves neither existence nor alignment; existence, forced asymmetry and selection are separate achievements', 'ledger_tier': 'D', 'evidence': 'P1 `HANDOFF_PROMPT.md` discipline sentence; STATEMENT.md', 'paper_sections': '3.5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:108', 'paper_locations': ['paper/tex/sec3_finding.tex:144', 'paper/tex/sec3_finding.tex:191'], 'declarations': ['R44.existence', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique']}, {'id': 'H11', 'statement': 'G11 instantiation for the registered system C_44, carried to Q by registration (A15) and the small-collar realization (R10): [H-G11-nonempty] = T1; [H-G11-hierarchy] (i) uniqueness at every scale = T8 (T1n, arbitrary geometrically nested registered families, `geometric_hierarchy_unique`), (ii) local forcing = Theorem 7.1 level by level (`parent_local_to_global`, R6, with `parent_atlas_eq_fine`, R7; radius growing with level; NOT local derivability from the unlabelled chair, N2), (iii) the period clause is reached by the halving tower (T13), not by per-period finite certificates, which the proof does not extract; conclusion = T2', 'ledger_tier': 'derived (T1n throughout: existence, hierarchy, local forcing, registration bridge and period exclusion are unconditional Lean theorems at the pin)', 'evidence': 'DISCOVERY.md §1b; T1, T2, T8, T13, R6, R7, R10, A15', 'paper_sections': '1.3, 3.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:109', 'paper_locations': ['paper/tex/sec1_intro.tex:226', 'paper/tex/sec1_intro.tex:232', 'paper/tex/sec3_finding.tex:144'], 'declarations': ['R44.existence', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique', 'R44.parent_atlas_eq_fine', 'R44.parent_local_to_global']}, {'id': 'H12', 'statement': 'Open problem: the same design test in other crystallographic settings and dimensions', 'ledger_tier': 'N', 'evidence': '—', 'paper_sections': '9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:110', 'paper_locations': ['paper/tex/sec9_remarks.tex:3'], 'declarations': []}, {'id': 'H13', 'statement': 'Disclosure of AI use ("Use of AI systems", after the acknowledgements): the solid was found by an OpenAI reasoning model (ChatGPT, Astra) given the Six Birds framework and the construction record (the framework was the input); the earlier construction record and the Lean/certificate work by automated agents (Codex implementer; Grok and Codex-on-a-second-model reviewers; two external adversarial rounds); the manuscript largely written by Claude Fable 5.1 under the author\'s direction and reviewed by Codex gpt-6-astra; no AI system is an author', 'ledger_tier': 'D', 'evidence': 'owner statement 2026-09-09; `LINEAGE.md` packet table; `provenance/HASH_CHAIN.md`; `lean/R44/MECHANIZATION_PLAN.md` agent roles; `paper/review/PROTOCOL.md`; `.codex/logs/reviewer/`', 'paper_sections': 'back matter, 3.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:111', 'paper_locations': ['paper/tex/main.tex:56', 'paper/tex/main.tex:67'], 'declarations': []}, {'id': 'H14', 'statement': "Provenance: the tile was landed in six hash-chained packets, each naming or embedding its predecessor by sha256, with the solid identical in packets 5 and 6; every packet's manifest and stdlib replay verified in the fusion; steps 2–3 of the discovery are packets 3–5, the alignment proof packet 6", 'ledger_tier': 'D', 'evidence': '`provenance/HASH_CHAIN.md`; `LINEAGE.md` (packet table)', 'paper_sections': '3.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:112', 'paper_locations': ['paper/tex/main.tex:67', 'paper/tex/sec3_finding.tex:106', 'paper/tex/sec3_finding.tex:132'], 'declarations': []}, {'id': 'A1', 'statement': 'Local finiteness (A-L2.1)', 'ledger_tier': 'T1', 'evidence': 'T11', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:118', 'paper_locations': ['paper/tex/sec4_companions.tex:3'], 'declarations': ['R44.local_finiteness']}, {'id': 'A2', 'statement': 'Cone / sector budgets (A-L2.2)', 'ledger_tier': 'T1n', 'evidence': '`cone_sector_budgets_holds` (Proved/)', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:119', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:34'], 'declarations': ['R44.cone_sector_budgets_holds', 'R44.local_finiteness']}, {'id': 'A3', 'statement': 'Complete dihedral list (A-L3.1)', 'ledger_tier': 'T1n (+T1 arithmetic)', 'evidence': '`complete_dihedral_list_holds` (Proved/; hooks `native_mesh_incidence_trace`, `carrier_coordinate_states_table`, R8/R10); `deviations_distinct`', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:120', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:60'], 'declarations': ['R44.DischargeCarrierStrata.carrier_coordinate_states_table', 'R44.DischargeMeshTrace.native_mesh_incidence_trace', 'R44.complete_dihedral_list_holds', 'R44.deviations_distinct', 'R44.local_finiteness', 'R44.mesh_angle_audit']}, {'id': 'A4', 'statement': 'Generic feature-edge point has exactly one complementary partner (A-L3.2)', 'ledger_tier': 'T1n', 'evidence': '`generic_feature_partner_holds` (Proved/; exceptional-vertex table hooks)', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:121', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:91'], 'declarations': ['R44.generic_feature_partner_holds', 'R44.local_finiteness']}, {'id': 'A5', 'statement': 'Circular-cone formula at L = 3/25 (A-L4.1, arithmetic half)', 'ledger_tier': 'T1', 'evidence': '`circular_cone_solid_angle_holds` (Proved/)', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:122', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:112'], 'declarations': ['R44.circular_cone_solid_angle_holds', 'R44.feature_circular_cone_containment_holds', 'R44.local_finiteness']}, {'id': 'A6', 'statement': 'Feature circular-cone containment (A-L4.1, geometric half); Ω(L) > 4π/3 ⟺ 8L² < 1', 'ledger_tier': 'T1n / T1', 'evidence': '`feature_circular_cone_containment_holds` (Proved/); `coneAngle_gt_four_pi_div_three_iff`', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:123', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:112'], 'declarations': ['R44.circular_cone_solid_angle_holds', 'R44.coneAngle_gt_four_pi_div_three_iff', 'R44.feature_circular_cone_containment_holds', 'R44.local_finiteness']}, {'id': 'A7', 'statement': 'One tile accompanies the whole feature graph (A-T4.2)', 'ledger_tier': 'T1n', 'evidence': '`connected_feature_companion_holds` (Proved/)', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:124', 'paper_locations': ['paper/tex/sec1_intro.tex:232', 'paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:139'], 'declarations': ['R44.connected_feature_companion_holds', 'R44.local_finiteness']}, {'id': 'A8', 'statement': 'Containment ⇒ equal features, edge-length rigidity (A-L4.3)', 'ledger_tier': 'T1n (formal proof by compactness, not the written length count)', 'evidence': '`feature_containment_rigidity_holds` (Proved/; with g ≠ h, L5X)', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:125', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:165'], 'declarations': ['R44.feature_containment_rigidity_holds', 'R44.local_finiteness']}, {'id': 'A9', 'statement': 'Companion pose ∈ signed permutations × (1/8)Z³ (A-C4.4)', 'ledger_tier': 'T1n', 'evidence': '`companion_pose_discrete_holds` (Proved/; side condition `mate_records_roles_nonempty`, R8)', 'paper_sections': '4', 'ledger_location': 'paper/CLAIMS_LEDGER.md:126', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:190'], 'declarations': ['R44.companion_pose_discrete_holds', 'R44.local_finiteness', 'R44.mate_records_roles_nonempty']}, {'id': 'A10', 'statement': 'Eighth-grid baseline overlap ⇒ core overlap (A-L5.1); the written estimate gives box side ≥ 21/200', 'ledger_tier': 'T1 for the overlap implication (`retained_core_overlap_holds`); T3 for the written quantitative estimate (A-L5.1, `proof/ALIGNMENT_PROOF.md`)', 'evidence': '`retained_core_overlap_holds` (Proved/); A-L5.1', 'paper_sections': '5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:127', 'paper_locations': ['paper/tex/sec5_registration.tex:3'], 'declarations': ['R44.retained_core_overlap_holds']}, {'id': 'A11', 'statement': 'Companion census: 6,862 formula poses; 1,545 collide with the root directly; 5,317 isolated (5,234 fractional, 83 integral); 5,273 rejected by companion-option collisions; 299,975 option collisions; 44 survivors equal to the atlas as a set; two independent Python implementations', 'ledger_tier': 'T1n / T2', 'evidence': '`mates_census`; `verify/packets/r44_unrestricted_alignment/results/alignment_verification.json` (`complete_feature_mates`); `verify/replay.py` (two implementations, `verify/packets/*`)', 'paper_sections': '5, App. B', 'ledger_location': 'paper/CLAIMS_LEDGER.md:128', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/appB_census.tex:25', 'paper/tex/notation.tex:3', 'paper/tex/sec1_intro.tex:232', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec5_registration.tex:3', 'paper/tex/sec5_registration.tex:26'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.retained_core_overlap_holds', 'R44.substitution_modular_coincidence']}, {'id': 'A12', 'statement': 'Only the 44 registered mates occur (A-L5.3)', 'ledger_tier': 'T1', 'evidence': '`only_registered_mates_holds` (Proved/)', 'paper_sections': '5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:129', 'paper_locations': ['paper/tex/sec5_registration.tex:3', 'paper/tex/sec5_registration.tex:58'], 'declarations': ['R44.only_registered_mates_holds', 'R44.retained_core_overlap_holds']}, {'id': 'A13', 'statement': "A feature component's baseline cells = Z³ (A-L6.1)", 'ledger_tier': 'T1n', 'evidence': '`baseline_component_covers_grid_holds` (Proved/)', 'paper_sections': '5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:130', 'paper_locations': ['paper/tex/sec5_registration.tex:3', 'paper/tex/sec5_registration.tex:81'], 'declarations': ['R44.baseline_component_covers_grid_holds', 'R44.retained_core_overlap_holds']}, {'id': 'A14', 'statement': "The component's tiles cover R³ (A-L6.2)", 'ledger_tier': 'T1 (formal proof by clamping to an interior point, not assembly-wide gluing)', 'evidence': '`component_solids_cover_holds` (Proved/)', 'paper_sections': '5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:131', 'paper_locations': ['paper/tex/sec5_registration.tex:3', 'paper/tex/sec5_registration.tex:103'], 'declarations': ['R44.component_solids_cover_holds', 'R44.retained_core_overlap_holds']}, {'id': 'A15', 'statement': 'Unrestricted alignment: every tiling is registered (A-T6.3)', 'ledger_tier': 'T1n', 'evidence': '`unrestricted_alignment_holds` (Proved/); `registration_of_tiling`', 'paper_sections': '5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:132', 'paper_locations': ['paper/tex/sec1_intro.tex:154', 'paper/tex/sec1_intro.tex:176', 'paper/tex/sec1_intro.tex:232', 'paper/tex/sec5_registration.tex:3', 'paper/tex/sec5_registration.tex:130', 'paper/tex/sec5_registration.tex:147'], 'declarations': ['R44.registration_of_tiling', 'R44.retained_core_overlap_holds', 'R44.unrestricted_alignment_holds']}, {'id': 'A16', 'statement': 'Tube construction: per-tube homeomorphisms, gluing, image = Q (A§1)', 'ledger_tier': 'T1n', 'evidence': '`per_tube_homeomorphisms_holds`, `feature_tube_maps_glue_holds`, `feature_tube_map_carries_carrier_holds` (Proved/); disjointness and boundary identity proved (T1)', 'paper_sections': '2, App.', 'ledger_location': 'paper/CLAIMS_LEDGER.md:133', 'paper_locations': ['paper/tex/sec2_solid.tex:125'], 'declarations': ['R44.feature_tube_map_carries_carrier_holds', 'R44.feature_tube_maps_glue_holds', 'R44.per_tube_homeomorphisms_holds']}, {'id': 'A17', 'statement': 'Errata E1, E3, E6 carried into the text', 'ledger_tier': 'T3 (text); the corrected lemmas are T1n/T1 (A8, A4, A14)', 'evidence': '`proof/ERRATA.md`', 'paper_sections': '4, 5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:134', 'paper_locations': ['paper/tex/sec4_companions.tex:3', 'paper/tex/sec4_companions.tex:60', 'paper/tex/sec4_companions.tex:91', 'paper/tex/sec4_companions.tex:165', 'paper/tex/sec5_registration.tex:103'], 'declarations': ['R44.complete_dihedral_list_holds', 'R44.component_solids_cover_holds', 'R44.deviations_distinct', 'R44.feature_containment_rigidity_holds', 'R44.generic_feature_partner_holds', 'R44.local_finiteness', 'R44.mesh_angle_audit']}, {'id': 'R1', 'statement': '21 internal contacts refine to a 30-state closure; 372 sign equations; 12 balanced components; canonical profile', 'ledger_tier': 'T1n', 'evidence': '`contact_closure_30`, `profile_canonical`', 'paper_sections': '3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:140', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec2_solid.tex:3', 'paper/tex/sec2_solid.tex:47'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.substitution_modular_coincidence']}, {'id': 'R2', 'statement': '2,388 shell poses in 48 frames → exactly the 44-contact atlas', 'ledger_tier': 'T1n', 'evidence': '`atlas_44`', 'paper_sections': '3, 5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:141', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/appB_census.tex:25', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec3_finding.tex:55'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.substitution_modular_coincidence']}, {'id': 'R3', 'statement': '19 contact frames generate the 24-element proper cubic group', 'ledger_tier': 'T1n', 'evidence': '`orientation_group_24`', 'paper_sections': '3, 7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:142', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec3_finding.tex:55', 'paper/tex/sec3_finding.tex:83'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.substitution_modular_coincidence']}, {'id': 'R4', 'statement': '33 first shells (exact covers of the 22-cell shell), each with one candidate parent signature', 'ledger_tier': 'T1n', 'evidence': '`first_shells_33`', 'paper_sections': '6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:143', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec6_hierarchy.tex:3'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.substitution_modular_coincidence']}, {'id': 'R5', 'statement': '14 of 32 outer-root shells completable, 18 not; centre typed central', 'ledger_tier': 'T1n', 'evidence': '`central_completion`', 'paper_sections': '6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:144', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec6_hierarchy.tex:3', 'paper/tex/sec6_hierarchy.tex:20'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.parent_local_to_global', 'R44.profile_canonical', 'R44.substitution_modular_coincidence', 'R44.unique_parent']}, {'id': 'R6', 'statement': 'All 28 parent pairs conflict ⇒ unique parent (T7.1); local-to-global proved', 'ledger_tier': 'T1n (unconditional)', 'evidence': '`parent_conflicts_28`; `registered_first_shells`, `certified_shells_complete_parents`, `complete_parent_conflicts`, `parent_local_to_global`', 'paper_sections': '6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:145', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec1_intro.tex:232', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec6_hierarchy.tex:3'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.certified_shells_complete_parents', 'R44.complete_parent_conflicts', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.parent_local_to_global', 'R44.profile_canonical', 'R44.registered_first_shells', 'R44.substitution_modular_coincidence']}, {'id': 'R7', 'statement': '697 → 116 → 44 parent contacts, all even; halved = fine atlas; coarsening preserves admissibility (T8.1); common parity; halved baseline tiling', 'ledger_tier': 'T1n (unconditional)', 'evidence': '`parent_atlas_eq_fine`; `parent_atlas_admissibility`, `parent_common_parity`, `halved_parent_baseline_tiling`', 'paper_sections': '6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:146', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec1_intro.tex:154', 'paper/tex/sec1_intro.tex:214', 'paper/tex/sec1_intro.tex:232', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec3_finding.tex:83', 'paper/tex/sec6_hierarchy.tex:3', 'paper/tex/sec6_hierarchy.tex:55'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.coarsening_is_tiling', 'R44.coarsening_translation_equivariant', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.halved_parent_baseline_tiling', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_admissibility', 'R44.parent_atlas_eq_fine', 'R44.parent_common_parity', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.small_collar_realization_holds', 'R44.substitution_modular_coincidence']}, {'id': 'R8', 'statement': 'Same-frame grandchild at (2,2,2); nested controls 64/4,096 poses, 448/28,672 cells', 'ledger_tier': 'T1n', 'evidence': '`nested_substitution_controls`, `hierarchy_cell_controls`', 'paper_sections': '6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:147', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec3_finding.tex:200', 'paper/tex/sec6_hierarchy.tex:3', 'paper/tex/sec6_hierarchy.tex:87'], 'declarations': ['R44.atlas_44', 'R44.carrier_hierarchy', 'R44.carrier_hierarchy_unique', 'R44.central_completion', 'R44.contact_closure_30', 'R44.existence', 'R44.first_shells_33', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique', 'R44.hierarchy_cell_controls', 'R44.mates_census', 'R44.nested_substitution_controls', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.small_collar_realization_holds', 'R44.substitution_modular_coincidence']}, {'id': 'R9', 'statement': 'Registered shell cell controls (192 features → 22 shell cells; legal frames among 48)', 'ledger_tier': 'T1n', 'evidence': '`registered_shell_cell_controls`', 'paper_sections': '5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:148', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec5_registration.tex:3'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.registered_shell_cell_controls', 'R44.retained_core_overlap_holds', 'R44.substitution_modular_coincidence']}, {'id': 'R10', 'statement': 'Small-collar realization: a baseline chair tiling with matching profiles is carried to a Q-tiling by the glued tubes (E2, E6)', 'ledger_tier': 'T1n (formal proof identifies duplicate tube sites by centre; separation 21/200)', 'evidence': '`small_collar_realization_holds` (Proved/)', 'paper_sections': '3, 6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:149', 'paper_locations': ['paper/tex/sec1_intro.tex:232', 'paper/tex/sec3_finding.tex:200'], 'declarations': ['R44.existence', 'R44.small_collar_realization_holds']}, {'id': 'R11', 'statement': 'Erratum E4 (half-integer coset after halving; periods still in Z³)', 'ledger_tier': 'T3', 'evidence': '`proof/ERRATA.md` E4', 'paper_sections': '7', 'ledger_location': 'paper/CLAIMS_LEDGER.md:150', 'paper_locations': ['paper/tex/sec6_hierarchy.tex:80'], 'declarations': []}, {'id': 'R12', 'statement': 'Erratum E5 (the native-asymmetry statement is split into carrier recovery, frame reduction and the 48-frame comparison; tiers as in O5: T1 / T1n / T1 at the pinned commit)', 'ledger_tier': 'D', 'evidence': '`proof/ERRATA.md` E5; O5', 'paper_sections': '2', 'ledger_location': 'paper/CLAIMS_LEDGER.md:151', 'paper_locations': ['paper/tex/sec2_solid.tex:92'], 'declarations': ['R44.planar_area_carrier_recovery_holds']}, {'id': 'L1', 'statement': 'The 168-label lattice substitution is total; matrix column sum 8; least primitive exponent 3; least modular-coincidence depth 3 at a=(0,0,2), i=78', 'ledger_tier': 'T1n / T2', 'evidence': '`substitution_modular_coincidence` (`Theorems.lean`, by declaration name); `verify/substitution_modular_coincidence.py`', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:157', 'paper_locations': ['paper/tex/appB_census.tex:3', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.substitution_modular_coincidence']}, {'id': 'L2', 'statement': 'The coincidence check ranges over all 168 starting labels (not diagonal)', 'ledger_tier': 'T2', 'evidence': 'replay: `states = [(ORIGIN, frozenset(range(168)))]`', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:158', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'L3', 'statement': 'A legal two-sided fixed point exists: a substitution-fixed labelling of Z³ is determined by its 2×2×2 seed block on {−1,0}³, each label fixed by its residue map; the substitution language contains exactly 27 fixed seed blocks, in proper-rotation orbits of sizes 24 and 3; every seed reproduces itself in place under iteration, so its iterates labell all of Z³ with every finite patch legal; every finite patch occurs inside a refinement of the native chair (primitivity), so the encoded chair poses form a registered chair tiling with all contacts in A₄₄ and small-collar realization gives a tiling T_w by Q. (Earlier draft: 24, from a level-3 search; corrected by the external manuscript review 2026-09-09, B1.)', 'ledger_tier': 'T1n (`seed_language_closure`: closure sizes, 27 seeds, orbits 24 + 3, histogram {1:24, 8:3}, in-place reproduction, frame commutation) / T2 / written (seed → tiling)', 'evidence': "`R44.seed_language_closure` (Theorems.lean at the pin; landed 6a7e7a4c9a); replay step `seed_language_closure` in `verify/replay.py`; `paper/scripts/seed_language_closure.py` (stdlib; reconstructs φ from `certificates/candidate_certificate.json` through `verify/substitution_modular_coincidence.py`; exact closure of the 2×2×2 block language 168 → 600 → 1,278 → 1,398 → 1,410, stable; `STATUS: PASS`); `paper/review/external_2026-09-09/` (reviewer's independent audit, same numbers)", 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:159', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:123', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.seed_language_closure', 'R44.substitution_modular_coincidence']}, {'id': 'L4', 'statement': 'PF eigenvalue 8 = \\|det 2I\\|; label classes of a fixed point partition Z³', 'ledger_tier': 'derived', 'evidence': 'column sum 8 (L1); L3', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:160', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:123', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.seed_language_closure', 'R44.substitution_modular_coincidence']}, {'id': 'L5', 'statement': 'Label classes are regular model sets with 2-adic internal space; pure point diffractive', 'ledger_tier': 'C', 'evidence': 'Q16 (iv)⇒(ii), Q17', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:161', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'L6', 'statement': 'The structure is limit-periodic (term of BMS98/BG10), not quasicrystalline; contrast with the hat', 'ledger_tier': 'C / Q', 'evidence': 'Q18, Q15', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:162', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'L7', 'statement': 'Level of the statement: fixed point; hull membership of every tiling open', 'ledger_tier': 'N', 'evidence': 'DECISIONS.md Q-A; T9', 'paper_sections': '8, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:163', 'paper_locations': ['paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'L8', 'statement': 'At depths 1, 2 and 3 the address-count distributions by possible-label-set size are {42:8}, {6:48, 24:16} and {1:336, 6:160, 24:16} (all addresses tracked from the full label set)', 'ledger_tier': 'T2', 'evidence': '`verify/substitution_modular_coincidence.py`, output field `coincidence_label_set_sizes`; `paper/scripts/coincidence_tree.py`', 'paper_sections': '8 (Fig. 8.1)', 'ledger_location': 'paper/CLAIMS_LEDGER.md:164', 'paper_locations': [], 'declarations': ['R44.substitution_modular_coincidence']}, {'id': 'L9', 'statement': 'Sym(T_w) is the stabilizer of the seed w in the 24-element proper cubic group G (acting on the seed cube by moving cells and composing frames): stabilizer orders 1 for the 24 seeds of the large orbit and 8 for the three seeds of the small orbit; for the seed (28,84,91,35,98,42,49,105) the stabilizer is generated by (x,y,z)↦(z,y,−x) and (x,y,z)↦(−x,−y,z). Deduction: a symmetry (R,t) has R ∈ G, t ∈ Z³ (T3 argument of Thm 7.2); parent recognition is rotation-covariant (children of a rotated pose are the rotated children, finite check) and translation-covariant (Thm 6.5), so D(gT_w) = (R,t/2) T_w = T_w forces t ∈ ∩ 2ⁿZ³ = 0; the substitution commutes with G on labelled cells (finite check), so R T_w = T_{Rw} and R T_w = T_w iff Rw = w. Hence tilings with trivial symmetry group and tilings with symmetry group of order 8 exist; Q is not strongly aperiodic in the CGGL24 sense', 'ledger_tier': 'T1n (`seed_language_closure`: orbits and stabilizer histogram) / T2 (generators, covariance 12,096 configuration and 576 pose checks) / written (deduction)', 'evidence': '`R44.seed_language_closure`; `paper/scripts/seed_language_closure.py` (stdlib; reconstructs φ from `certificates/candidate_certificate.json` through `verify/substitution_modular_coincidence.py`; exact closure of the 2×2×2 block language 168 → 600 → 1,278 → 1,398 → 1,410, stable; `STATUS: PASS`); external review 2026-09-09 M1 (independent argument via intrinsic ancestor classes, `paper/review/external_2026-09-09/COUNTEREXAMPLES.md`)', 'paper_sections': '8, 7, 9, 1.1, 1.5', 'ledger_location': 'paper/CLAIMS_LEDGER.md:165', 'paper_locations': ['paper/tex/sec8_mechanization.tex:123'], 'declarations': ['R44.seed_language_closure', 'R44.substitution_modular_coincidence']}, {'id': 'L10', 'statement': 'Non-exhausting hierarchies occur: the fixed point with seed (6,0,4,5,2,3,0,1) has two whole tiles in frame A = diag(−1,−1,1), at the origin (its own child 000 at every level; ancestors 2ⁿAP, union the closed octant {x ≤ 0, y ≤ 0, z ≥ 0}) and at (1,1,−1) (its own central child; ancestors fill the closure of the complement): an infinite fault surface on supertile boundaries at all levels, inside a substitution-legal tiling; its symmetry group is trivial (L9). Non-exhaustion is not exclusion from the substitution hull', 'ledger_tier': 'T2 (poses, self-roles, stabilizer) / written (ancestor union)', 'evidence': '`paper/scripts/seed_language_closure.py` (stdlib; reconstructs φ from `certificates/candidate_certificate.json` through `verify/substitution_modular_coincidence.py`; exact closure of the 2×2×2 block language 168 → 600 → 1,278 → 1,398 → 1,410, stable; `STATUS: PASS`) (w0 block: poses, self-roles [0] and [7], stabilizer 1); external review 2026-09-09 M1', 'paper_sections': '6, 8, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:166', 'paper_locations': ['paper/tex/sec6_hierarchy.tex:123', 'paper/tex/sec8_mechanization.tex:123'], 'declarations': ['R44.seed_language_closure', 'R44.substitution_modular_coincidence']}, {'id': 'M1', 'statement': '23 finite theorems in Lean; method per theorem (`decide` / `native_decide`)', 'ledger_tier': 'T1/T1n', 'evidence': '`lean/R44/AXIOMS.md` decision-method ledger; `Theorems.lean`', 'paper_sections': '8, App. C', 'ledger_location': 'paper/CLAIMS_LEDGER.md:172', 'paper_locations': ['paper/tex/appC_lean.tex:3', 'paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:164', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec8_mechanization.tex:5'], 'declarations': []}, {'id': 'M2', 'statement': '`r44_einstein` axiom line: `propext, Classical.choice, Quot.sound` plus 21 named `native_decide` hooks: the 15 of the phase-1 finite theorems (atlas_44, central_completion, contact_closure_30, first_shells_33, hierarchy_cell_controls, mates_census, mesh_angle_audit, nested_substitution_controls, orientation_group_24, parent_atlas_eq_fine, parent_conflicts_28, profile_canonical, registered_shell_cell_controls, solid_mesh_exact, transported_role_geometry; the last is an auxiliary check in `Proved/CarrierFeatureFrameReduction.lean`, not one of the 23 finite theorems of `Theorems.lean`, so the hooks cover the finite computations used by the proof including that auxiliary check) and six closed checks over certified data made by the discharge proofs (mate_records_roles_nonempty R8; native_mesh_incidence_trace R8; carrier_coordinate_states_table R10; feature_index_bounds, feature_vertex_lookup, carrier_vertex_lookup); no `sorryAx` anywhere', 'ledger_tier': 'T1n (unconditional)', 'evidence': "the `'R44.r44_einstein' depends on axioms` line of `lean/R44/build_axioms.log` (at the pinned commit; verbatim in APPENDIX_DATA.md §5)", 'paper_sections': '8, App. C', 'ledger_location': 'paper/CLAIMS_LEDGER.md:173', 'paper_locations': ['paper/tex/appC_lean.tex:3', 'paper/tex/appC_lean.tex:36', 'paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:164', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec6_hierarchy.tex:108', 'paper/tex/sec8_mechanization.tex:5', 'paper/tex/sec8_mechanization.tex:69'], 'declarations': ['R44.DischargeCarrierStrata.carrier_coordinate_states_table', 'R44.DischargeMeshTrace.native_mesh_incidence_trace', 'R44.DischargeVertexData.carrier_vertex_lookup', 'R44.DischargeVertexData.feature_index_bounds', 'R44.DischargeVertexData.feature_vertex_lookup', 'R44.atlas_44', 'R44.carrierHierarchy_geometric', 'R44.carrier_hierarchy_unique', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique', 'R44.hierarchy_cell_controls', 'R44.mate_records_roles_nonempty', 'R44.mates_census', 'R44.mesh_angle_audit', 'R44.nested_substitution_controls', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.r44_einstein', 'R44.registered_shell_cell_controls', 'R44.solid_mesh_exact']}, {'id': 'M2b', 'statement': 'CLOSED at the re-pin: no admission remains; `R44/Discharge/` holds no module; the three declarations that carried `sorryAx` at dd2735d9bd are admission-free theorems in `Proved/`; the paper says so where it mentions the discharge programme (historical)', 'ledger_tier': 'D', 'evidence': '`lean/R44/AXIOMS.md` at d90313a717 (`sorryAx` nowhere); controls.sh `admissions=0`', 'paper_sections': '8, App. C', 'ledger_location': 'paper/CLAIMS_LEDGER.md:174', 'paper_locations': ['paper/tex/appC_lean.tex:3', 'paper/tex/appC_lean.tex:36', 'paper/tex/sec8_mechanization.tex:17'], 'declarations': ['R44.DischargeCarrierStrata.carrier_coordinate_states_table', 'R44.DischargeMeshTrace.native_mesh_incidence_trace', 'R44.carrier_hierarchy', 'R44.geometric_hierarchy_unique', 'R44.mate_records_roles_nonempty', 'R44.r44_einstein', 'R44.r44_einstein_of_hypotheses', 'R44.tiling_homochiral']}, {'id': 'M3', 'statement': 'Two independent Python implementations of the companion census; stdlib only; replay ≈ 1 minute', 'ledger_tier': 'T2', 'evidence': '`verify/replay.py`; `verify/packets/r44_unrestricted_alignment`, `verify/packets/einstein_macrostate`', 'paper_sections': '8, App. A', 'ledger_location': 'paper/CLAIMS_LEDGER.md:175', 'paper_locations': ['paper/tex/appA_data.tex:3', 'paper/tex/appA_data.tex:24', 'paper/tex/appB_census.tex:3', 'paper/tex/main.tex:20', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec5_registration.tex:3', 'paper/tex/sec5_registration.tex:26', 'paper/tex/sec8_mechanization.tex:42'], 'declarations': ['R44.atlas_44', 'R44.central_completion', 'R44.contact_closure_30', 'R44.first_shells_33', 'R44.mates_census', 'R44.orientation_group_24', 'R44.parent_atlas_eq_fine', 'R44.parent_conflicts_28', 'R44.profile_canonical', 'R44.retained_core_overlap_holds', 'R44.substitution_modular_coincidence']}, {'id': 'M4', 'statement': 'No floating-point value enters any verified predicate: certificates are integer/`Fraction` data, checkers use `int` and `fractions.Fraction`, Lean decides integer literals; decimal numbers occur only as timing metadata (`"seconds"` fields, `time.monotonic()`) and in the OBJ rendering export; the paper-lane verification scripts named in Appendix A likewise use `int`/`Fraction` only, the planar-area checker bounding sqrt q by (q + m²)/(2m) with integer-square-root witnesses m, exactly', 'ledger_tier': 'D', 'evidence': '`certificates/candidate_certificate.json` and `companion_collision_certificate.json` (one `"seconds"` decimal each, no other decimal literal); `verify/tube_formula_controls.py:11`, `verify/packets/*/src/*.py` (`from fractions import Fraction`); `verify/packets/einstein_macrostate/src/build_solid.py:64` (the only `float(`); `Theorems.lean` (`decide`/`native_decide`)', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:176', 'paper_locations': ['paper/tex/appA_data.tex:3', 'paper/tex/appA_data.tex:24', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec8_mechanization.tex:42'], 'declarations': []}, {'id': 'M5', 'statement': 'Negative controls: must-fail files with checked diagnostics; scope regressions; six corrupted inputs rejected', 'ledger_tier': 'T2', 'evidence': '`lean/R44/scripts/controls.sh`; `verify/packets/r44_unrestricted_alignment/src/test_mutations.py`', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:177', 'paper_locations': ['paper/tex/appA_data.tex:24', 'paper/tex/appC_lean.tex:3', 'paper/tex/appC_lean.tex:20', 'paper/tex/appC_lean.tex:36', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec8_mechanization.tex:42'], 'declarations': ['R44.DischargeCarrierStrata.carrier_coordinate_states_table', 'R44.DischargeMeshTrace.native_mesh_incidence_trace', 'R44.mate_records_roles_nonempty', 'R44.r44_einstein', 'R44.r44_einstein_of_hypotheses']}, {'id': 'M6', 'statement': 'Lean 4.31.0 + Mathlib (commit at freeze); build time and peak RSS from the gate log', 'ledger_tier': 'D', 'evidence': 'APPENDIX_DATA.md (step 8)', 'paper_sections': 'App. C', 'ledger_location': 'paper/CLAIMS_LEDGER.md:178', 'paper_locations': ['paper/tex/appA_data.tex:3', 'paper/tex/appA_data.tex:24', 'paper/tex/sec8_mechanization.tex:69'], 'declarations': []}, {'id': 'M7', 'statement': '"Code." paragraph: repository, tag, license', 'ledger_tier': 'D', 'evidence': 'step 18', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:179', 'paper_locations': ['paper/tex/appA_data.tex:3', 'paper/tex/appA_data.tex:86', 'paper/tex/sec1_intro.tex:255', 'paper/tex/sec8_mechanization.tex:69'], 'declarations': []}, {'id': 'M9', 'statement': '`carrier_hierarchy_unique` quantifies over `LevelSupertilePose H T n`, whose `mem` field places every level-n pose in `(coarseningIterate H n T).placements` = Dⁿ(T); CLOSED at the re-pin (formal bridge landed 44282b63e8): `GeometricHierarchy` carries raw poses with no membership; `geometric_hierarchy_canonical` proves every level-n pose lies in Dⁿ(T) by induction from unique parents; `geometric_hierarchy_unique` gives uniqueness among arbitrary geometrically nested registered families; `carrierHierarchy_geometric` the existence half; must-fail `negative/NoncanonicalGeometricHierarchy.lean`', 'ledger_tier': 'D', 'evidence': 'pinned `LogicalSpine.lean` (LevelSupertilePose, carrier_hierarchy_unique, GeometricHierarchy, geometric_hierarchy_canonical, geometric_hierarchy_unique, carrierHierarchy_geometric); external review 2026-09-09 M2; `paper/review/external_2026-09-09/REVIEW.md`', 'paper_sections': '6, 8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:181', 'paper_locations': ['paper/tex/sec6_hierarchy.tex:108'], 'declarations': ['R44.carrierHierarchy_geometric', 'R44.carrier_hierarchy_unique', 'R44.existence', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique']}, {'id': 'N1', 'statement': 'Not every tiling is shown to lie in the substitution hull; non-exhaustion (L10) is not exclusion from the hull', 'ledger_tier': '', 'evidence': '', 'paper_sections': '6, 8, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:187', 'paper_locations': ['paper/tex/sec1_intro.tex:154', 'paper/tex/sec6_hierarchy.tex:3', 'paper/tex/sec6_hierarchy.tex:87', 'paper/tex/sec6_hierarchy.tex:123', 'paper/tex/sec8_mechanization.tex:105', 'paper/tex/sec8_mechanization.tex:123', 'paper/tex/sec8_mechanization.tex:190'], 'declarations': ['R44.carrier_hierarchy', 'R44.carrier_hierarchy_unique', 'R44.first_shells_33', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique', 'R44.seed_language_closure', 'R44.substitution_modular_coincidence']}, {'id': 'N2', 'statement': 'Local derivability from the unlabelled chair is not claimed', 'ledger_tier': '', 'evidence': '', 'paper_sections': '6', 'ledger_location': 'paper/CLAIMS_LEDGER.md:188', 'paper_locations': ['paper/tex/sec1_intro.tex:154', 'paper/tex/sec6_hierarchy.tex:3', 'paper/tex/sec6_hierarchy.tex:87', 'paper/tex/sec6_hierarchy.tex:123'], 'declarations': ['R44.carrier_hierarchy', 'R44.carrier_hierarchy_unique', 'R44.first_shells_33', 'R44.geometric_hierarchy_canonical', 'R44.geometric_hierarchy_unique']}, {'id': 'N3', 'statement': 'No claim that the bound 24 is attained, nor which symmetry groups occur beyond the orders 1 and 8 of L9; every-tiling triviality (CGGL24 strong aperiodicity) is false for Q (L9), not open', 'ledger_tier': '', 'evidence': '', 'paper_sections': '1.1, 1.5, 7, 9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:189', 'paper_locations': ['paper/tex/sec1_intro.tex:80', 'paper/tex/sec7_aperiodicity.tex:3', 'paper/tex/sec9_remarks.tex:3'], 'declarations': ['R44.no_period', 'R44.period_halving', 'R44.r44_einstein', 'R44.sym_card_le_24']}, {'id': 'N4', 'statement': 'No minimality (faces, vertices, features) and no convexity', 'ledger_tier': '', 'evidence': '', 'paper_sections': '9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:190', 'paper_locations': ['paper/tex/sec1_intro.tex:98', 'paper/tex/sec9_remarks.tex:3', 'paper/tex/sec9_remarks.tex:44'], 'declarations': []}, {'id': 'N5', 'statement': 'No decidability result', 'ledger_tier': '', 'evidence': '', 'paper_sections': '9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:191', 'paper_locations': ['paper/tex/sec9_remarks.tex:3'], 'declarations': []}, {'id': 'N6', 'statement': 'No perturbation stability beyond R§11', 'ledger_tier': '', 'evidence': '', 'paper_sections': '9', 'ledger_location': 'paper/CLAIMS_LEDGER.md:192', 'paper_locations': ['paper/tex/sec9_remarks.tex:3'], 'declarations': []}, {'id': 'N7', 'statement': 'No second, computer-free proof', 'ledger_tier': '', 'evidence': '', 'paper_sections': '1.3', 'ledger_location': 'paper/CLAIMS_LEDGER.md:193', 'paper_locations': ['paper/tex/sec1_intro.tex:232'], 'declarations': []}, {'id': 'N8', 'statement': 'No mathlib coverage; the theorem is checked modulo the 21 named `native_decide` hooks, not by kernel `decide` alone', 'ledger_tier': '', 'evidence': '', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:194', 'paper_locations': ['paper/tex/sec1_intro.tex:164', 'paper/tex/sec8_mechanization.tex:5'], 'declarations': []}, {'id': 'N9', 'statement': 'The theorem-free periodicity search (`simulations/periodicity_search.py`; receipt `simulations/periodicity_search_report.json`, commit d90313a717) is evidence, not proof, and is used nowhere in the proof: grid-registered copies (48 frames), full-rank sublattices of index ≤ 40 (43,981, all UNSAT), coronas to radius 5 of 5 (251 copies), wall time 108,682 s; positive controls (unit cube index 1, featureless chair index 7) receipted in `simulations/review/2026-09-09_periodicity_search_r1_eddy.md`', 'ledger_tier': '', 'evidence': '', 'paper_sections': '8', 'ledger_location': 'paper/CLAIMS_LEDGER.md:195', 'paper_locations': ['paper/tex/sec8_mechanization.tex:80'], 'declarations': []}], 'axiom_records': {'R44.children_partition_2P': {'tier': 'T1', 'axioms': [], 'supplied_record': "'R44.children_partition_2P' does not depend on any axioms", 'location': 'lean/R44/build_axioms.log:1', 'declaration_locations': ['lean/R44/R44/Theorems.lean:10'], 'location_resolution': 'source declaration'}, 'R44.contact_closure_30': {'tier': 'T1n', 'axioms': ['R44.contact_closure_30._native.native_decide.ax_1_1'], 'supplied_record': "'R44.contact_closure_30' depends on axioms: [R44.contact_closure_30._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:2', 'declaration_locations': ['lean/R44/R44/Theorems.lean:15'], 'location_resolution': 'source declaration'}, 'R44.profile_canonical': {'tier': 'T1n', 'axioms': ['propext', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.profile_canonical' depends on axioms: [propext, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:3', 'declaration_locations': ['lean/R44/R44/Theorems.lean:22'], 'location_resolution': 'source declaration'}, 'R44.atlas_44': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1'], 'supplied_record': "'R44.atlas_44' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:4', 'declaration_locations': ['lean/R44/R44/Theorems.lean:27'], 'location_resolution': 'source declaration'}, 'R44.first_shells_33': {'tier': 'T1n', 'axioms': ['propext', 'R44.first_shells_33._native.native_decide.ax_1_1'], 'supplied_record': "'R44.first_shells_33' depends on axioms: [propext, R44.first_shells_33._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:5', 'declaration_locations': ['lean/R44/R44/Theorems.lean:32'], 'location_resolution': 'source declaration'}, 'R44.central_completion': {'tier': 'T1n', 'axioms': ['propext', 'R44.central_completion._native.native_decide.ax_1_1'], 'supplied_record': "'R44.central_completion' depends on axioms: [propext, R44.central_completion._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:6', 'declaration_locations': ['lean/R44/R44/Theorems.lean:37'], 'location_resolution': 'source declaration'}, 'R44.parent_conflicts_28': {'tier': 'T1n', 'axioms': ['R44.parent_conflicts_28._native.native_decide.ax_1_1'], 'supplied_record': "'R44.parent_conflicts_28' depends on axioms: [R44.parent_conflicts_28._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:7', 'declaration_locations': ['lean/R44/R44/Theorems.lean:42'], 'location_resolution': 'source declaration'}, 'R44.parent_atlas_eq_fine': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1'], 'supplied_record': "'R44.parent_atlas_eq_fine' depends on axioms: [propext, Classical.choice, Quot.sound, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:8', 'declaration_locations': ['lean/R44/R44/Theorems.lean:48'], 'location_resolution': 'source declaration'}, 'R44.mates_census': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mates_census._native.native_decide.ax_1_1'], 'supplied_record': "'R44.mates_census' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mates_census._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:9', 'declaration_locations': ['lean/R44/R44/Theorems.lean:55'], 'location_resolution': 'source declaration'}, 'R44.deviations_distinct': {'tier': 'T1', 'axioms': [], 'supplied_record': "'R44.deviations_distinct' does not depend on any axioms", 'location': 'lean/R44/build_axioms.log:10', 'declaration_locations': ['lean/R44/R44/Theorems.lean:60'], 'location_resolution': 'source declaration'}, 'R44.no_native_symmetry': {'tier': 'T1', 'axioms': ['propext'], 'supplied_record': "'R44.no_native_symmetry' depends on axioms: [propext]", 'location': 'lean/R44/build_axioms.log:11', 'declaration_locations': ['lean/R44/R44/Theorems.lean:66'], 'location_resolution': 'source declaration'}, 'R44.native_features_length': {'tier': 'T1', 'axioms': ['propext'], 'supplied_record': "'R44.native_features_length' depends on axioms: [propext]", 'location': 'lean/R44/build_axioms.log:12', 'declaration_locations': ['lean/R44/R44/Theorems.lean:71'], 'location_resolution': 'source declaration'}, 'R44.native_feature_centers_nodup': {'tier': 'T1', 'axioms': ['propext'], 'supplied_record': "'R44.native_feature_centers_nodup' depends on axioms: [propext]", 'location': 'lean/R44/build_axioms.log:13', 'declaration_locations': ['lean/R44/R44/Theorems.lean:76'], 'location_resolution': 'source declaration'}, 'R44.native_feature_normal_axis': {'tier': 'T1', 'axioms': ['propext'], 'supplied_record': "'R44.native_feature_normal_axis' depends on axioms: [propext]", 'location': 'lean/R44/build_axioms.log:14', 'declaration_locations': ['lean/R44/R44/Theorems.lean:83'], 'location_resolution': 'source declaration'}, 'R44.solid_mesh_exact': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.solid_mesh_exact._native.native_decide.ax_1_1'], 'supplied_record': "'R44.solid_mesh_exact' depends on axioms: [propext, Classical.choice, Quot.sound, R44.solid_mesh_exact._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:15', 'declaration_locations': ['lean/R44/R44/Theorems.lean:93'], 'location_resolution': 'source declaration'}, 'R44.boundary_sphere': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.boundary_sphere._native.native_decide.ax_1_1'], 'supplied_record': "'R44.boundary_sphere' depends on axioms: [propext, Classical.choice, Quot.sound, R44.boundary_sphere._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:16', 'declaration_locations': ['lean/R44/R44/Theorems.lean:106'], 'location_resolution': 'source declaration'}, 'R44.nested_substitution_controls': {'tier': 'T1n', 'axioms': ['propext', 'Quot.sound', 'R44.nested_substitution_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.nested_substitution_controls' depends on axioms: [propext, Quot.sound, R44.nested_substitution_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:17', 'declaration_locations': ['lean/R44/R44/Theorems.lean:119'], 'location_resolution': 'source declaration'}, 'R44.hierarchy_cell_controls': {'tier': 'T1n', 'axioms': ['R44.hierarchy_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.hierarchy_cell_controls' depends on axioms: [R44.hierarchy_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:18', 'declaration_locations': ['lean/R44/R44/Theorems.lean:125'], 'location_resolution': 'source declaration'}, 'R44.registered_shell_cell_controls': {'tier': 'T1n', 'axioms': ['propext', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.registered_shell_cell_controls' depends on axioms: [propext, R44.registered_shell_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:19', 'declaration_locations': ['lean/R44/R44/Theorems.lean:131'], 'location_resolution': 'source declaration'}, 'R44.orientation_group_24': {'tier': 'T1n', 'axioms': ['propext', 'Quot.sound', 'R44.orientation_group_24._native.native_decide.ax_1_1'], 'supplied_record': "'R44.orientation_group_24' depends on axioms: [propext, Quot.sound, R44.orientation_group_24._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:20', 'declaration_locations': ['lean/R44/R44/Theorems.lean:136'], 'location_resolution': 'source declaration'}, 'R44.substitution_modular_coincidence': {'tier': 'T1n', 'axioms': ['propext', 'R44.substitution_modular_coincidence._native.native_decide.ax_1_1'], 'supplied_record': "'R44.substitution_modular_coincidence' depends on axioms: [propext, R44.substitution_modular_coincidence._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:21', 'declaration_locations': ['lean/R44/R44/Theorems.lean:148'], 'location_resolution': 'source declaration'}, 'R44.seed_language_closure': {'tier': 'T1n', 'axioms': ['propext', 'Quot.sound', 'R44.seed_language_closure._native.native_decide.ax_1_1'], 'supplied_record': "'R44.seed_language_closure' depends on axioms: [propext, Quot.sound, R44.seed_language_closure._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:22', 'declaration_locations': ['lean/R44/R44/Theorems.lean:166'], 'location_resolution': 'source declaration'}, 'R44.mesh_angle_audit': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mesh_angle_audit._native.native_decide.ax_1_1'], 'supplied_record': "'R44.mesh_angle_audit' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mesh_angle_audit._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:23', 'declaration_locations': ['lean/R44/R44/Theorems.lean:182'], 'location_resolution': 'source declaration'}, 'R44.no_self_mate': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.no_self_mate' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:24', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:769'], 'location_resolution': 'source declaration'}, 'R44.unitCube_compact': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.unitCube_compact' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:25', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:77'], 'location_resolution': 'source declaration'}, 'R44.unitCube_regularClosed': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.unitCube_regularClosed' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:26', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:102'], 'location_resolution': 'source declaration'}, 'R44.unitCube_interior_nonempty': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.unitCube_interior_nonempty' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:27', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:110'], 'location_resolution': 'source declaration'}, 'R44.P_compact': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.P_compact' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:28', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:161'], 'location_resolution': 'source declaration'}, 'R44.P_regularClosed': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.P_regularClosed' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:29', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:165'], 'location_resolution': 'source declaration'}, 'R44.P_interior_nonempty': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.P_interior_nonempty' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:30', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:169'], 'location_resolution': 'source declaration'}, 'R44.cellCenter_mem_interior_unitCube': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.cellCenter_mem_interior_unitCube' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:31', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:183'], 'location_resolution': 'source declaration'}, 'R44.featureTubeMap_fixed_on_boundary': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.featureTubeMap_fixed_on_boundary' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:32', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:455'], 'location_resolution': 'source declaration'}, 'R44.featureTubeSupport_coordinate_bound': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.featureTubeSupport_coordinate_bound' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:33', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:345'], 'location_resolution': 'source declaration'}, 'R44.featureTubeSupports_pairwiseDisjoint': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.featureTubeSupports_pairwiseDisjoint' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:34', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:415'], 'location_resolution': 'source declaration'}, 'R44.planar_area_native_reduction': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.planar_area_native_reduction' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:35', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:824'], 'location_resolution': 'source declaration'}, 'R44.hasTubeHomeomorphism_of_tube_construction': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.hasTubeHomeomorphism_of_tube_construction' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:36', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:950'], 'location_resolution': 'source declaration'}, 'R44.P_homeomorphic_Q_of_tube_homeomorphism': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.P_homeomorphic_Q_of_tube_homeomorphism' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:37', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:960'], 'location_resolution': 'source declaration'}, 'R44.Q_compact_of_tube_homeomorphism': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.Q_compact_of_tube_homeomorphism' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:38', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:976'], 'location_resolution': 'source declaration'}, 'R44.Q_regularClosed_of_tube_homeomorphism': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.Q_regularClosed_of_tube_homeomorphism' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:39', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:983'], 'location_resolution': 'source declaration'}, 'R44.Q_interior_nonempty_of_tube_homeomorphism': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.Q_interior_nonempty_of_tube_homeomorphism' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:40', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:989'], 'location_resolution': 'source declaration'}, 'R44.compactRegularClosedBall_of_tube_homeomorphism': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.compactRegularClosedBall_of_tube_homeomorphism' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:41', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:997'], 'location_resolution': 'source declaration'}, 'R44.tube_homeomorphism': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.tube_homeomorphism' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:42', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2062'], 'location_resolution': 'source declaration'}, 'R44.tube_ball': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.tube_ball' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:43', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2066'], 'location_resolution': 'source declaration'}, 'R44.local_finiteness': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.local_finiteness' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:44', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1095'], 'location_resolution': 'source declaration'}, 'R44.strictDownwardCone_isOpen': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.strictDownwardCone_isOpen' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:45', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1117'], 'location_resolution': 'source declaration'}, 'R44.strictDownwardCone_subset_tangentCone_hypograph': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.strictDownwardCone_subset_tangentCone_hypograph' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:46', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1127'], 'location_resolution': 'source declaration'}, 'R44.circularCone_isOpen': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.circularCone_isOpen' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:47', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1163'], 'location_resolution': 'source declaration'}, 'R44.coneAngle_gt_four_pi_div_three_iff': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.coneAngle_gt_four_pi_div_three_iff' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:48', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1169'], 'location_resolution': 'source declaration'}, 'R44.solidAngle_linearIsometry_image': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.solidAngle_linearIsometry_image' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:49', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1198'], 'location_resolution': 'source declaration'}, 'R44.solidAngle_mono': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.solidAngle_mono' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:50', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1218'], 'location_resolution': 'source declaration'}, 'R44.r44_cone_angle_bound': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.r44_cone_angle_bound' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:51', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2078'], 'location_resolution': 'source declaration'}, 'R44.uniform_solid_angle': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.uniform_solid_angle' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:52', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2092'], 'location_resolution': 'source declaration'}, 'R44.native_asymmetry_planar_reduction': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.transported_role_geometry._native.native_decide.ax_1_1'], 'supplied_record': "'R44.native_asymmetry_planar_reduction' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1, R44.transported_role_geometry._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:53', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2071'], 'location_resolution': 'source declaration'}, 'R44.native_asymmetry_reduction': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.native_asymmetry_reduction' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:54', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:875'], 'location_resolution': 'source declaration'}, 'R44.retained_core_overlap_holds': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.retained_core_overlap_holds' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:55', 'declaration_locations': ['lean/R44/R44/Proved/RetainedCoreOverlap.lean:283'], 'location_resolution': 'source declaration'}, 'R44.carrier_feature_frame_reduction_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.transported_role_geometry._native.native_decide.ax_1_1'], 'supplied_record': "'R44.carrier_feature_frame_reduction_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1, R44.transported_role_geometry._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:56', 'declaration_locations': ['lean/R44/R44/Proved/CarrierFeatureFrameReduction.lean:280'], 'location_resolution': 'source declaration'}, 'R44.circular_cone_solid_angle_holds': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.circular_cone_solid_angle_holds' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:57', 'declaration_locations': ['lean/R44/R44/Proved/CircularConeSolidAngle.lean:366'], 'location_resolution': 'source declaration'}, 'R44.per_tube_homeomorphisms_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.per_tube_homeomorphisms_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:58', 'declaration_locations': ['lean/R44/R44/Proved/PerTubeHomeomorphisms.lean:173'], 'location_resolution': 'source declaration'}, 'R44.feature_tube_maps_glue_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.feature_tube_maps_glue_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:59', 'declaration_locations': ['lean/R44/R44/Proved/FeatureTubeMapsGlue.lean:23'], 'location_resolution': 'source declaration'}, 'R44.feature_tube_map_carries_carrier_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.feature_tube_map_carries_carrier_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:60', 'declaration_locations': ['lean/R44/R44/Proved/FeatureTubeMapCarriesCarrier.lean:29'], 'location_resolution': 'source declaration'}, 'R44.feature_circular_cone_containment_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.feature_circular_cone_containment_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:61', 'declaration_locations': ['lean/R44/R44/Proved/FeatureCircularConeContainment.lean:164'], 'location_resolution': 'source declaration'}, 'R44.planar_area_carrier_recovery_holds': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.planar_area_carrier_recovery_holds' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:62', 'declaration_locations': ['lean/R44/R44/Proved/PlanarAreaCarrierRecovery.lean:115'], 'location_resolution': 'source declaration'}, 'R44.mate_records_roles_nonempty': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1'], 'supplied_record': "'R44.mate_records_roles_nonempty' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:63', 'declaration_locations': ['lean/R44/R44/Proved/MateCensusSemantics.lean:24'], 'location_resolution': 'source declaration'}, 'R44.cone_sector_budgets_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.cone_sector_budgets_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:64', 'declaration_locations': ['lean/R44/R44/Proved/ConeSectorBudgets.lean:160'], 'location_resolution': 'source declaration'}, 'R44.connected_feature_companion_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.connected_feature_companion_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:65', 'declaration_locations': ['lean/R44/R44/Proved/ConnectedFeatureCompanion.lean:173'], 'location_resolution': 'source declaration'}, 'R44.feature_containment_rigidity_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.feature_containment_rigidity_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:66', 'declaration_locations': ['lean/R44/R44/Proved/FeatureContainmentRigidity.lean:121'], 'location_resolution': 'source declaration'}, 'R44.companion_pose_discrete_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1'], 'supplied_record': "'R44.companion_pose_discrete_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:67', 'declaration_locations': ['lean/R44/R44/Proved/CompanionPoseDiscrete.lean:19'], 'location_resolution': 'source declaration'}, 'R44.only_registered_mates_holds': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.only_registered_mates_holds' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:68', 'declaration_locations': ['lean/R44/R44/Proved/OnlyRegisteredMates.lean:24'], 'location_resolution': 'source declaration'}, 'R44.component_solids_cover_holds': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.component_solids_cover_holds' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:69', 'declaration_locations': ['lean/R44/R44/Proved/ComponentSolidsCover.lean:21'], 'location_resolution': 'source declaration'}, 'R44.small_collar_realization_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.small_collar_realization_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:70', 'declaration_locations': ['lean/R44/R44/Proved/SmallCollarRealization.lean:79'], 'location_resolution': 'source declaration'}, 'R44.DischargeAssembly.registered_assembly_collar_data': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeAssembly.registered_assembly_collar_data' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:71', 'declaration_locations': ['lean/R44/R44/Proved/AssemblyCollarData.lean:50'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.DischargeCarrierStrata.carrier_coordinate_states_table': {'tier': 'T1n', 'axioms': ['propext', 'Quot.sound', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeCarrierStrata.carrier_coordinate_states_table' depends on axioms: [propext, Quot.sound, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:72', 'declaration_locations': ['lean/R44/R44/Proved/CarrierCoordinateStates.lean:128'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.DischargeMeshTrace.native_mesh_incidence_trace': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeMeshTrace.native_mesh_incidence_trace' depends on axioms: [propext, Classical.choice, Quot.sound, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:73', 'declaration_locations': ['lean/R44/R44/Proved/MeshChartIncidence.lean:91'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.DischargeVertexData.feature_index_bounds': {'tier': 'T1n', 'axioms': ['R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeVertexData.feature_index_bounds' depends on axioms: [R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:74', 'declaration_locations': ['lean/R44/R44/Proved/NativeExceptionalVertices.lean:221'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.DischargeVertexData.feature_vertex_lookup': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeVertexData.feature_vertex_lookup' depends on axioms: [propext, Classical.choice, Quot.sound, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:75', 'declaration_locations': ['lean/R44/R44/Proved/NativeExceptionalVertices.lean:225'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.DischargeVertexData.carrier_vertex_lookup': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeVertexData.carrier_vertex_lookup' depends on axioms: [propext, Classical.choice, Quot.sound, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:76', 'declaration_locations': ['lean/R44/R44/Proved/NativeExceptionalVertices.lean:331'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.DischargeMeridian.native_oriented_wedge_charts': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeMeridian.native_oriented_wedge_charts' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:77', 'declaration_locations': ['lean/R44/R44/Proved/NativeDihedralSectors.lean:23'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.DischargeNativeStrata.native_boundary_strata': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.DischargeNativeStrata.native_boundary_strata' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:78', 'declaration_locations': ['lean/R44/R44/Proved/NativeBoundaryStrata.lean:71'], 'location_resolution': 'source declaration inside namespace (name match; inspect namespace at source)'}, 'R44.complete_dihedral_list_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1'], 'supplied_record': "'R44.complete_dihedral_list_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:79', 'declaration_locations': ['lean/R44/R44/Proved/CompleteDihedralList.lean:410'], 'location_resolution': 'source declaration'}, 'R44.generic_feature_partner_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.generic_feature_partner_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.profile_canonical._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:80', 'declaration_locations': ['lean/R44/R44/Proved/GenericFeaturePartner.lean:67'], 'location_resolution': 'source declaration'}, 'R44.baseline_component_covers_grid_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.baseline_component_covers_grid_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:81', 'declaration_locations': ['lean/R44/R44/Proved/BaselineComponentCoversGrid.lean:142'], 'location_resolution': 'source declaration'}, 'R44.unrestricted_alignment_holds': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.unrestricted_alignment_holds' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:82', 'declaration_locations': ['lean/R44/R44/Proved/UnrestrictedAlignment.lean:57'], 'location_resolution': 'source declaration'}, 'R44.registration_of_tiling': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.registration_of_tiling' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:83', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6261'], 'location_resolution': 'source declaration'}, 'R44.handedness_eq_one_or_neg_one': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.handedness_eq_one_or_neg_one' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:84', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7096'], 'location_resolution': 'source declaration'}, 'R44.handedness_mul': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.handedness_mul' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:85', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7111'], 'location_resolution': 'source declaration'}, 'R44.Registration.handedness_eq': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.Registration.handedness_eq' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:86', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7141'], 'location_resolution': 'source declaration'}, 'R44.tiling_homochiral': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.tiling_homochiral' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:87', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7157'], 'location_resolution': 'source declaration'}, 'R44.tiling_orientation_normalization': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.tiling_orientation_normalization' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:88', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7164'], 'location_resolution': 'source declaration'}, 'R44.tiling_handedness_dichotomy': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.tiling_handedness_dichotomy' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:89', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7172'], 'location_resolution': 'source declaration'}, 'R44.no_mixed_handedness': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.no_mixed_handedness' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:90', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7185'], 'location_resolution': 'source declaration'}, 'R44.tiling_chirality_corollary': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.tiling_chirality_corollary' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:91', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7194'], 'location_resolution': 'source declaration'}, 'R44.parent_local_to_global': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.parent_local_to_global' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:92', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6270'], 'location_resolution': 'source declaration'}, 'R44.unique_parent': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.unique_parent' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:93', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6288'], 'location_resolution': 'source declaration'}, 'R44.Packing.ext': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.Packing.ext' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:94', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:490'], 'location_resolution': 'source declaration'}, 'R44.Tiling.ext': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.Tiling.ext' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:95', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:497'], 'location_resolution': 'source declaration'}, 'R44.realizesPose_unique': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.realizesPose_unique' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:96', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:784'], 'location_resolution': 'source declaration'}, 'R44.childOfParent_unique': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.childOfParent_unique' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:97', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:54'], 'location_resolution': 'source declaration'}, 'R44.ParentPartition.ext': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.ParentPartition.ext' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:98', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:128'], 'location_resolution': 'source declaration'}, 'R44.hasUniqueParent_of_completeParents': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.hasUniqueParent_of_completeParents' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:99', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:141'], 'location_resolution': 'source declaration'}, 'R44.halfPose_mul': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.halfPose_mul' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:100', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:248'], 'location_resolution': 'source declaration'}, 'R44.halfPose_translate': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.halfPose_translate' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:101', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:265'], 'location_resolution': 'source declaration'}, 'R44.halfPose_realizes_halvePose': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.halfPose_realizes_halvePose' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:102', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:288'], 'location_resolution': 'source declaration'}, 'R44.computedMacroLegal_even': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.computedMacroLegal_even' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:103', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:324'], 'location_resolution': 'source declaration'}, 'R44.halfPose_realizes_doubledPose': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.halfPose_realizes_doubledPose' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:104', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:3167'], 'location_resolution': 'source declaration'}, 'R44.halvedParentPoses_translate': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.halvedParentPoses_translate' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:105', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:337'], 'location_resolution': 'source declaration'}, 'R44.closedContacts_subset_legal': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.closedContacts_subset_legal' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:106', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:481'], 'location_resolution': 'source declaration'}, 'R44.hierarchyPatch_one': {'tier': 'T1', 'axioms': ['propext', 'Quot.sound'], 'supplied_record': "'R44.hierarchyPatch_one' depends on axioms: [propext, Quot.sound]", 'location': 'lean/R44/build_axioms.log:107', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:588'], 'location_resolution': 'source declaration'}, 'R44.hierarchyPatch_two': {'tier': 'T1', 'axioms': ['propext', 'Quot.sound'], 'supplied_record': "'R44.hierarchyPatch_two' depends on axioms: [propext, Quot.sound]", 'location': 'lean/R44/build_axioms.log:108', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:590'], 'location_resolution': 'source declaration'}, 'R44.hierarchy_patch_nesting': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.hierarchy_patch_nesting' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:109', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:610'], 'location_resolution': 'source declaration'}, 'R44.cells_refine': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.cells_refine' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:110', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:1247'], 'location_resolution': 'source declaration'}, 'R44.cells_two_refine': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.cells_two_refine' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:111', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:1340'], 'location_resolution': 'source declaration'}, 'R44.hierarchy_cells_disjoint': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.hierarchy_cells_disjoint' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:112', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:1925'], 'location_resolution': 'source declaration'}, 'R44.hierarchy_box_containment': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.hierarchy_box_containment' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:113', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:1953'], 'location_resolution': 'source declaration'}, 'R44.hierarchy_boxes_exhaust': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.hierarchy_boxes_exhaust' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:114', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2006'], 'location_resolution': 'source declaration'}, 'R44.nested_contact_language': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.nested_contact_language' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:115', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:3673'], 'location_resolution': 'source declaration'}, 'R44.nested_baseline_assembly': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.nested_baseline_assembly' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:116', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:4013'], 'location_resolution': 'source declaration'}, 'R44.nested_atlas_baseline_assembly': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.hierarchy_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.nested_atlas_baseline_assembly' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.hierarchy_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:117', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:4074'], 'location_resolution': 'source declaration'}, 'R44.formula_mate_owns_far_cell': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.formula_mate_owns_far_cell' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:118', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:987', 'lean/R44/R44/Proved/ConcreteCompanions.lean:129'], 'location_resolution': 'source declaration'}, 'R44.body_cellCenter_mem_interior': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.body_cellCenter_mem_interior' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:119', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:1041', 'lean/R44/R44/Proved/RegisteredCellGeometry.lean:143'], 'location_resolution': 'source declaration'}, 'R44.component_cell_owner_unique': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.component_cell_owner_unique' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:120', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:1057'], 'location_resolution': 'source declaration'}, 'R44.perm_lt_three': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.perm_lt_three' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:121', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1416'], 'location_resolution': 'source declaration'}, 'R44.frameCoordinate_of_lt': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.frameCoordinate_of_lt' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:122', 'declaration_locations': ['lean/R44/R44/LogicalSpineFoundation.lean:1424'], 'location_resolution': 'source declaration'}, 'R44.frameActReal_mul': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.frameActReal_mul' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:123', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:688', 'lean/R44/R44/PoseAlgebra.lean:39'], 'location_resolution': 'source declaration'}, 'R44.frameActReal_transpose': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.frameActReal_transpose' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:124', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:732', 'lean/R44/R44/PoseAlgebra.lean:76'], 'location_resolution': 'source declaration'}, 'R44.realizesPose_relative': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.realizesPose_relative' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:125', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:826', 'lean/R44/R44/PoseAlgebra.lean:250'], 'location_resolution': 'source declaration'}, 'R44.role_companion_owns_far_cell': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.role_companion_owns_far_cell' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:126', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2127'], 'location_resolution': 'source declaration'}, 'R44.shellCell_unique_owner_with_index': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.shellCell_unique_owner_with_index' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:127', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2157'], 'location_resolution': 'source declaration'}, 'R44.enumerateShells_complete': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.enumerateShells_complete' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:128', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2269'], 'location_resolution': 'source declaration'}, 'R44.realizesPose_pose_unique': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.realizesPose_pose_unique' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:129', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2430', 'lean/R44/R44/PoseAlgebra.lean:151'], 'location_resolution': 'source declaration'}, 'R44.featureAdjacent_comm': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.featureAdjacent_comm' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:130', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2456'], 'location_resolution': 'source declaration'}, 'R44.sameFeatureComponent_symm': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.sameFeatureComponent_symm' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:131', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2462'], 'location_resolution': 'source declaration'}, 'R44.sameFeatureComponent_trans': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.sameFeatureComponent_trans' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:132', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2469'], 'location_resolution': 'source declaration'}, 'R44.shellOwner_indices_compatible': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.shellOwner_indices_compatible' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:133', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2512'], 'location_resolution': 'source declaration'}, 'R44.mem_realizedShellCover': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.mem_realizedShellCover' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:134', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2611'], 'location_resolution': 'source declaration'}, 'R44.realizedShellCover_exact': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.realizedShellCover_exact' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:135', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2638'], 'location_resolution': 'source declaration'}, 'R44.realizedShellCover_compatible': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.realizedShellCover_compatible' depends on axioms: [propext, Classical.choice, Quot.sound, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:136', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2667'], 'location_resolution': 'source declaration'}, 'R44.registered_first_shells': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.registered_first_shells' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:137', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:2679'], 'location_resolution': 'source declaration'}, 'R44.realizesPose_transform': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.realizesPose_transform' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:138', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:4272', 'lean/R44/R44/PoseAlgebra.lean:311'], 'location_resolution': 'source declaration'}, 'R44.touch_relative_has_shell_cell': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1'], 'supplied_record': "'R44.touch_relative_has_shell_cell' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:139', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:4135'], 'location_resolution': 'source declaration'}, 'R44.certified_shells_complete_parents': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.certified_shells_complete_parents' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:140', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:4651'], 'location_resolution': 'source declaration'}, 'R44.complete_parent_conflicts': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.complete_parent_conflicts' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:141', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6229'], 'location_resolution': 'source declaration'}, 'R44.actual_parent_crossContacts_legal': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.actual_parent_crossContacts_legal' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:142', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:4855'], 'location_resolution': 'source declaration'}, 'R44.parent_atlas_admissibility': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.parent_atlas_admissibility' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:143', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6037'], 'location_resolution': 'source declaration'}, 'R44.evenPose_transform': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.evenPose_transform' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:144', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:3133'], 'location_resolution': 'source declaration'}, 'R44.evenPose_relative_trans': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.evenPose_relative_trans' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:145', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:3142'], 'location_resolution': 'source declaration'}, 'R44.parent_common_parity': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.parent_common_parity' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:146', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:5695'], 'location_resolution': 'source declaration'}, 'R44.halved_parent_baseline_tiling': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.halved_parent_baseline_tiling' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:147', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:5936'], 'location_resolution': 'source declaration'}, 'R44.uniqueParentPartition_translate': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.uniqueParentPartition_translate' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:148', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6298'], 'location_resolution': 'source declaration'}, 'R44.coarsening_is_tiling': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.coarsening_is_tiling' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:149', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6327'], 'location_resolution': 'source declaration'}, 'R44.coarsening_realization': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.coarsening_realization' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:150', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6312'], 'location_resolution': 'source declaration'}, 'R44.coarsening_placements': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.coarsening_placements' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:151', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6339'], 'location_resolution': 'source declaration'}, 'R44.coarsening_translation_equivariant': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.coarsening_translation_equivariant' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:152', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6346'], 'location_resolution': 'source declaration'}, 'R44.period_halving': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.transported_role_geometry._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.period_halving' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.transported_role_geometry._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:153', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6404'], 'location_resolution': 'source declaration'}, 'R44.period_iterate': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.transported_role_geometry._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.period_iterate' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.transported_role_geometry._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:154', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6984'], 'location_resolution': 'source declaration'}, 'R44.carrier_hierarchy_exists': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.carrier_hierarchy_exists' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:155', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6769'], 'location_resolution': 'source declaration'}, 'R44.carrier_hierarchy_unique': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.carrier_hierarchy_unique' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:156', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6862'], 'location_resolution': 'source declaration'}, 'R44.carrier_hierarchy': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.carrier_hierarchy' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:157', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6967'], 'location_resolution': 'source declaration'}, 'R44.carrierHierarchy_geometric': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.carrierHierarchy_geometric' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:158', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6797'], 'location_resolution': 'source declaration'}, 'R44.geometric_hierarchy_canonical': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.geometric_hierarchy_canonical' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:159', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6821'], 'location_resolution': 'source declaration'}, 'R44.geometric_hierarchy_unique': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.geometric_hierarchy_unique' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:160', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6937'], 'location_resolution': 'source declaration'}, 'R44.CarrierHierarchy.toGeometric_toCarrier': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.CarrierHierarchy.toGeometric_toCarrier' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:161', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6946'], 'location_resolution': 'source declaration'}, 'R44.GeometricHierarchy.toCarrier_toGeometric': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.GeometricHierarchy.toCarrier_toGeometric' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:162', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6953'], 'location_resolution': 'source declaration'}, 'R44.gridVector_norm_separated': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.gridVector_norm_separated' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:163', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:6993'], 'location_resolution': 'source declaration'}, 'R44.no_period': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.no_period' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:164', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7029'], 'location_resolution': 'source declaration'}, 'R44.sym_card_le_24': {'tier': 'T1', 'axioms': ['propext', 'Classical.choice', 'Quot.sound'], 'supplied_record': "'R44.sym_card_le_24' depends on axioms: [propext, Classical.choice, Quot.sound]", 'location': 'lean/R44/build_axioms.log:165', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7059'], 'location_resolution': 'source declaration'}, 'R44.nested_exhaustion': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.hierarchy_cell_controls._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.nested_exhaustion' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.hierarchy_cell_controls._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:166', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7069'], 'location_resolution': 'source declaration'}, 'R44.existence': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.contact_closure_30._native.native_decide.ax_1_1', 'R44.hierarchy_cell_controls._native.native_decide.ax_1_1', 'R44.nested_substitution_controls._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1'], 'supplied_record': "'R44.existence' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.contact_closure_30._native.native_decide.ax_1_1, R44.hierarchy_cell_controls._native.native_decide.ax_1_1, R44.nested_substitution_controls._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:167', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7081'], 'location_resolution': 'source declaration'}, 'R44.r44_einstein_of_hypotheses': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.contact_closure_30._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.hierarchy_cell_controls._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.nested_substitution_controls._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.transported_role_geometry._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.r44_einstein_of_hypotheses' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.contact_closure_30._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.hierarchy_cell_controls._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.nested_substitution_controls._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.transported_role_geometry._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:168', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7208'], 'location_resolution': 'source declaration'}, 'R44.r44_einstein': {'tier': 'T1n', 'axioms': ['propext', 'Classical.choice', 'Quot.sound', 'R44.atlas_44._native.native_decide.ax_1_1', 'R44.central_completion._native.native_decide.ax_1_1', 'R44.contact_closure_30._native.native_decide.ax_1_1', 'R44.first_shells_33._native.native_decide.ax_1_1', 'R44.hierarchy_cell_controls._native.native_decide.ax_1_1', 'R44.mate_records_roles_nonempty._native.native_decide.ax_1_1', 'R44.mates_census._native.native_decide.ax_1_1', 'R44.mesh_angle_audit._native.native_decide.ax_1_1', 'R44.nested_substitution_controls._native.native_decide.ax_1_1', 'R44.orientation_group_24._native.native_decide.ax_1_1', 'R44.parent_atlas_eq_fine._native.native_decide.ax_1_1', 'R44.parent_conflicts_28._native.native_decide.ax_1_1', 'R44.profile_canonical._native.native_decide.ax_1_1', 'R44.registered_shell_cell_controls._native.native_decide.ax_1_1', 'R44.solid_mesh_exact._native.native_decide.ax_1_1', 'R44.transported_role_geometry._native.native_decide.ax_1_1', 'R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1', 'R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1', 'R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1', 'R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1'], 'supplied_record': "'R44.r44_einstein' depends on axioms: [propext, Classical.choice, Quot.sound, R44.atlas_44._native.native_decide.ax_1_1, R44.central_completion._native.native_decide.ax_1_1, R44.contact_closure_30._native.native_decide.ax_1_1, R44.first_shells_33._native.native_decide.ax_1_1, R44.hierarchy_cell_controls._native.native_decide.ax_1_1, R44.mate_records_roles_nonempty._native.native_decide.ax_1_1, R44.mates_census._native.native_decide.ax_1_1, R44.mesh_angle_audit._native.native_decide.ax_1_1, R44.nested_substitution_controls._native.native_decide.ax_1_1, R44.orientation_group_24._native.native_decide.ax_1_1, R44.parent_atlas_eq_fine._native.native_decide.ax_1_1, R44.parent_conflicts_28._native.native_decide.ax_1_1, R44.profile_canonical._native.native_decide.ax_1_1, R44.registered_shell_cell_controls._native.native_decide.ax_1_1, R44.solid_mesh_exact._native.native_decide.ax_1_1, R44.transported_role_geometry._native.native_decide.ax_1_1, R44.DischargeCarrierStrata.carrier_coordinate_states_table._native.native_decide.ax_1_1, R44.DischargeMeshTrace.native_mesh_incidence_trace._native.native_decide.ax_1_1, R44.DischargeVertexData.carrier_vertex_lookup._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_index_bounds._native.native_decide.ax_1_1, R44.DischargeVertexData.feature_vertex_lookup._native.native_decide.ax_1_1]", 'location': 'lean/R44/build_axioms.log:169', 'declaration_locations': ['lean/R44/R44/LogicalSpine.lean:7231'], 'location_resolution': 'source declaration'}}}
session = Session(PIN, CLAIM_MAP)
session.prepare()

## 1. What is the shape?

**T2 · display only.** Drag to turn the tile. Select **Diagram**, then **Use true
scale**, to see the actual feature footprints and heights. Try **Periodic
cousin** and **Try a true period** to explore the featureless chair.

The embedded viewer is the repository's existing viewer. Its source hashes are
checked before it is shown. The next cell runs the mesh and finite certificate
checks; their complete output is saved with your session.

In [ ]:
session.run("viewer", session.viewer, "T2 source data; display only")

In [ ]:
session.replay()

## 2. Why do the tiny features matter?

**T2 · exact collision witness.** This example shows two copies implicated in
a rejection witness and the retained-core overlap box in yellow. Rotate it to
inspect the overlap. Coordinates below the picture are exact fractions.

Choose a different record (0–5272) and rerun the cell. `partner_index` selects
a companion within that record. The original packet's integer checker runs
first; only the resulting display coordinates are converted to floats.

In [ ]:
record_index = 0  # @param {type:"integer"}
partner_index = 0  # @param {type:"integer"}
session.collision(record_index, partner_index)

## 3. Which neighbours can fit?

**T2 · finite companion census.** Expand the fresh report below. The companion
census and the physical collision-box replay have separate checks. Their
scope statements describe exactly what each computation establishes.

In [ ]:
session.companions()

## 4. Why must tiles form larger groups?

**T2 · parent-completion certificates.** Select a shell (0–32). The central
tile and its recorded neighbours are shown together; the recorded parent role
and contact indices are printed above them.

This v1 displays certificate data and replays its checks. Recognizing a patch
you supply is a future U2 integration, not an operation of this notebook.

In [ ]:
shell_index = 0  # @param {type:"integer"}
session.parent(shell_index)

## 5. Why does the argument keep working?

**T2 · compare the actual contact sets.** This executes the registered checker
and displays both set differences, rather than reading a precomputed Boolean.
Expand the result to inspect all contacts yourself.

In [ ]:
session.atlas()

## Try to break a certificate

**T2 · six mutations.** Run the cell to test the six existing corruption controls.
Select a result to inspect its rejection. Each corruption is confined to a
temporary copy; the original inputs are checked again afterwards.

In [ ]:
mutations = session.mutations()
from IPython.display import Markdown
for result in mutations["tests"]:
    display(Markdown("**" + result["mutation"] + "** — `" + result["last_error_line"] + "`"))

**T2 · your turn.** Choose one corruption and an index, then run this cell.
The index addresses a triangle, a feature, a rejection witness, or a legal
contact, depending on the selected corruption. The unchanged checker decides
the result; an unexpected error is reported as a failed experiment.

In [ ]:
mutation_name = "missing_triangle"  # @param ["missing_triangle", "reversed_triangle", "changed_geometric_apex", "missing_rejection_witness", "wrong_companion_quantifier", "missing_registered_contact"]
mutation_index = 0  # @param {type:"integer"}
session.mutation(mutation_name, mutation_index)

## Try something that really repeats

**T2-adjacent · evidence only, used nowhere in the proof** (paper §8.1).
Run the periodic unit-cube and featureless-chair controls, then inspect their
explicit witnesses. This installs the optional `python-sat` dependency if
needed. It does not run the long R44 periodicity search.

In [ ]:
session.controls()

## 6. Does this settle the infinite problem?

**T1 / T1n / T3 · read the argument and its dependencies.** The map below is
generated from the paper's ledger and the supplied axiom log. Each declaration
has its own axiom-derived tier. Expand a source to read its exact proof text.
Finite experiments above do not execute the infinite-space argument.

In [ ]:
session.sources()

## Check the formal proof

**T1 / T1n · separate execution environment.** The logs above are supplied
records, not a fresh Lean build. A full build needs roughly 16 GB RAM and a
multi-gigabyte dependency download; budget 30 minutes or more after dependencies.

[Open the prepared Codespace](https://codespaces.new/ioannist/six-birds-tiles)
· [Read the build instructions](https://github.com/ioannist/six-birds-tiles/blob/main/notebook/LEAN.md)

Opening the environment does not launch the build. The instructions give one
command that builds the pinned sources, runs controls, compares fresh axiom
output with the supplied log, and writes a separate execution receipt.

## Keep your results

**T2 · session record.** Download your unsigned receipt to keep or give to your
AI. It includes the source hashes, host, date, timings, failures and skipped
steps. This Python notebook always records `lean_ran: false`.

In [ ]:
session.download()